# Segmentation client non-supervisée — Pipeline ML

**Objectif** : découvrir des **profils comportementaux** parmi les clients `ACTIVE` à partir de leurs données d'usage et de crédit, **sans étiquette de churn** et **sans score métier pré-calculé**.

Le résultat est un segment (`cluster_id` + `cluster_name`) et un flag d'anomalie (`anomaly_flag`) exploitables par un agent décisionnel LangGraph.

---

## Logique du pipeline

| Étape | Raison d'être | Sortie |
|---|---|---|
| **EDA** | Diagnostiquer ce qui rend les données difficiles (zero-inflation, skewness, collinéarité) **avant** de modéliser | Checklist de décisions pour la prépa |
| **Nettoyage** | Réparer les problèmes structurels (types, doublons, NaN) + winsoriser les outliers extrêmes | `df_clean` |
| **Feature engineering** | Créer de la **variance exploitable** là où les features brutes sont quasi-constantes (ratios, flags binaires, log) | ~15 features au lieu de 4 |
| **Périmètre ACTIVE** | Les STATE non-ACTIVE (DISCO/SUSPENDED/ON-HOLD) sont déjà segmentés par l'état administratif → aucune valeur ajoutée à les clusteriser | clients ACTIVE du dataset étudié |
| **Prépa modèle** | `PowerTransformer` (Yeo-Johnson) + scaler robuste pour redresser les distributions | matrice `X` |
| **Choix de K** | Croiser **4 métriques** (inertie, silhouette, DB, CH) + contrainte "aucun cluster < 0.5%" | K* |
| **Clustering** | Comparer **MiniBatchKMeans**, **GaussianMixture**, **BisectingKMeans** pour choisir ce qui matche la vraie structure | labels |
| **Interprétation** | Nommer chaque cluster automatiquement depuis ses **features les plus déviantes** (z-score vs population) | `cluster_name` |
| **Anomalies** | Isolation Forest sur le même pipeline de features | `anomaly_flag` |

---

## Principes de conception

1. **Segmentation comportementale non supervisée** — aucune étiquette, aucun score précalculé en entrée du clustering.
2. **Interprétabilité par construction** — chaque cluster est nommé via les features réelles qui le distinguent, pas via un score externe.
3. **Honnêteté sur les limites** — à chaque étape on mesure ce qui marche et ce qui ne marche pas (pas de silhouette trompeuse).

## 1. Imports et configuration

- `sklearn.preprocessing.PowerTransformer` (Yeo-Johnson) a été choisi car il accepte les valeurs nulles et positives, contrairement à Box-Cox qui exige des valeurs strictement positives. Il est adapté aux distributions asymétriques observées dans les variables financières.
- `MiniBatchKMeans` pour tenir à l'échelle une montée en charge si le volume augmente.
- `GaussianMixture` pour comparaison (clustering probabiliste).
- `BisectingKMeans` (hiérarchique descendant, plus stable que KMeans sur données déséquilibrées).
- `IsolationForest` pour la détection d'anomalies.

In [1]:
# Sécurisation de l'exécution locale Windows / VS Code.
# Ces variables limitent les conflits de threads BLAS/OpenMP avec scikit-learn.
#on limite à 1 thread pour que l’exécution soit plus stable.
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

In [2]:
from pathlib import Path
import os #sert à gérer les chemins des fichiers proprement.
import warnings #sert à masquer les messages d’avertissement Python.

import numpy as np #sert aux calculs numériques :
import pandas as pd
import matplotlib.pyplot as plt #sert à créer des graphiques
import seaborn as sns  #basé sur matplotlib, mais il donne des graphiques plus beaux et plus lisibles.

from sklearn.preprocessing import PowerTransformer, RobustScaler 
#sert à transformer les variables numériques pour rendre leurs distributions plus équilibrées.
#Le transformeur Yeo-Johnson permet de réduire cette asymétrie.
#RobustScaler sert à normaliser les données. RobustScaler remet les variables sur une échelle comparable.
#RobustScaler au lieu de StandardScaler RobustScaler est moins sensible aux valeurs extrêmes.


from sklearn.decomposition import PCA #Elle sert à réduire le nombre de variables tout en gardant le maximum d’information.
from sklearn.cluster import MiniBatchKMeans, BisectingKMeans
#MiniBatchKMeans est une version optimisée de KMeans.

from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore") #ne pas afficher les avertissements non critiques.
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environnement prêt.")

Environnement prêt.


## 2. Chargement brut

On lit le CSV **tel quel** — pas de filtrage, pas de conversion, juste un constat de ce qu'on a entre les mains.

In [3]:

from pathlib import Path
import os
import pandas as pd
#Cette partie permet de gérer automatiquement les chemins du projet de manière robuste.
# Chemins reproductibles : le notebook peut être lancé depuis la racine du projet
# ou depuis le dossier machine_learning/notebooks.


# Récupère le dossier courant depuis lequel le notebook est lancé
CURRENT_DIR = Path.cwd()

#On suppose au départ que le dossier courant est la racine du projet
PROJECT_DIR = CURRENT_DIR

# Cas 1 : le notebook est lancé directement depuis le dossier machine_learning
if PROJECT_DIR.name == "machine_learning":
    ML_DIR = PROJECT_DIR

# Cas 2 : le notebook est lancé depuis la racine du projet    
elif (PROJECT_DIR / "machine_learning").exists():
    ML_DIR = PROJECT_DIR / "machine_learning"
# Cas 3 : le notebook est lancé depuis un sous-dossier    
else:
    # Fallback utile si le notebook est lancé depuis un sous-dossier.
    candidates = [p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / "machine_learning").exists()]

        # Si aucun dossier machine_learning n’est trouvé, on arrête le programme
    if not candidates:
        raise FileNotFoundError("Impossible de localiser le dossier machine_learning. Lance le notebook depuis la racine du projet.")
   
    # On prend le premier dossier parent valide
    PROJECT_DIR = candidates[0]
    ML_DIR = PROJECT_DIR / "machine_learning"
# Chemin vers le dataset final utilisé pour le Machine Learning
DATA_PATH = Path(os.environ.get('BAD_DEBTS_MERGED_FILE', r'C:\\Users\\benab\\OneDrive\\Bureau\\PFE TT\\PLATEFORME TT\\backend\\runtime_data\\ml_imports\\28\\merged_dataset_inner.csv'))
# Dossier où seront enregistrés les résultats exportés
EXPORT_DIR = Path(os.environ.get('BAD_DEBTS_EXPORT_DIR', r'C:\\Users\\benab\\OneDrive\\Bureau\\PFE TT\\PLATEFORME TT\\backend\\runtime_data\\ml_imports\\28'))
#Cette ligne crée le dossier exports s’il n’existe pas encore.
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Projet     : {PROJECT_DIR}")
print(f"ML_DIR     : {ML_DIR}")
print(f"Fichier lu : {DATA_PATH}")
print(f"Shape      : {df_raw.shape[0]:,} lignes x {df_raw.shape[1]} colonnes")

print("\nColonnes :")
for c in df_raw.columns:
    print(f"  - {c:30s} {df_raw[c].dtype}")

df_raw.head(3)

Projet     : C:\Users\benab\OneDrive\Bureau\PFE TT\PLATEFORME TT\machine_learning
ML_DIR     : C:\Users\benab\OneDrive\Bureau\PFE TT\PLATEFORME TT\machine_learning
Fichier lu : C:\Users\benab\OneDrive\Bureau\PFE TT\PLATEFORME TT\backend\runtime_data\ml_imports\28\merged_dataset_inner.csv
Shape      : 9,748 lignes x 14 colonnes

Colonnes :
  - MSISDN                         int64
  - STATE_IN                       object
  - SUBSCRIBER_TYPE_IN             object
  - RATE_PLAN                      object
  - ACCOUNT_ACTIVATED_DATE         object
  - AVG_CREDIT_AMOUNT              float64
  - AVG_CREDIT_FEE                 float64
  - AVG_REIMBURSED_AMOUNT          float64
  - AVG_FEE_REIMBURSED             float64
  - AVG_REIMBURSE_RATIO            float64
  - AVG_DAYS_SINCE_CREDIT          float64
  - TOTAL_OUTSTANDING_AMOUNT       float64
  - TOTAL_OUTSTANDING_FEE          float64
  - NB_SOS                         int64


## 3. EDA — Diagnostic avant modélisation

**Principe** : on ne nettoie pas et on ne clusterise pas "à l'aveugle". Avant tout, on mesure ce qui va poser problème pour les algorithmes de clustering :

1. **Types & valeurs manquantes** — savoir ce qu'il faut convertir/imputer.
2. **Distributions** — détecter skewness et zero-inflation (deux ennemis du KMeans).
3. **Corrélations** — repérer la collinéarité (redondance qui biaise la distance).
4. **Variabilité par STATE** — voir quelles features portent réellement de l'info.
5. **Clusterisabilité** — estimer s'il existe vraiment des groupes dans les données, ou si on a un point massif et rien d'autre.

### 3.1 Types, manquants, doublons

In [4]:
diag = pd.DataFrame({
    "dtype":      df_raw.dtypes.astype(str),
    "n_missing":  df_raw.isna().sum(),
    "pct_missing": (df_raw.isna().sum() / len(df_raw) * 100).round(2),
    "n_unique":   df_raw.nunique(dropna=True),
})
print(f"Lignes totales   : {len(df_raw):,}")
print(f"Doublons MSISDN  : {df_raw['MSISDN'].duplicated().sum():,}")
print(f"Lignes distinctes: {df_raw.drop_duplicates().shape[0]:,}")
diag

Lignes totales   : 9,748
Doublons MSISDN  : 0
Lignes distinctes: 9,748


### 3.2 Distribution de la cible descriptive `STATE_IN`

`STATE_IN` n'est pas une cible supervisée, mais c'est la **variable administrative de référence** — elle nous servira plus tard à interpréter les clusters et à circonscrire le périmètre de clustering.

In [5]:
#compte combien de clients il y a dans chaque état + les valeurs manquantes.
state_counts = df_raw["STATE_IN"].value_counts(dropna=False)

#le pourcentage de chaque état
state_pct = (state_counts / len(df_raw) * 100).round(2)
#creation de tableau de résumé
state_summary = pd.DataFrame({"count": state_counts, "pct": state_pct})
print(state_summary)

#visualisation de la répartition des clients par état
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(state_counts.index[::-1], state_counts.values[::-1], color="steelblue")
for i, v in enumerate(state_counts.values[::-1]):
    ax.text(v, i, f" {v:,}  ({state_pct.iloc[::-1].iloc[i]}%)", va="center", fontsize=9)
ax.set_title("Répartition STATE_IN")
ax.set_xscale("log")
plt.tight_layout(); plt.show()

                      count    pct
STATE_IN                          
ACTIVE                 9208 94.460
SUSPENDED               497  5.100
Disconnected Network     28  0.290
ON-HOLD                  15  0.150


### 3.3 Statistiques et skewness des features numériques

Pour chaque feature on calcule :

- **skewness** : si `|skew| > 1` → distribution fortement asymétrique (mauvaise pour KMeans tel quel)
- **kurtosis** : outliers extrêmes si `> 3`
- **% de zéros** : signal de "zero-inflation" (point masse à 0 qui peut figer un cluster entier)

In [6]:
NUMERIC_RAW = [
    "AVG_CREDIT_AMOUNT", "AVG_CREDIT_FEE",
    "AVG_REIMBURSED_AMOUNT", "AVG_FEE_REIMBURSED",
    "AVG_REIMBURSE_RATIO", "AVG_DAYS_SINCE_CREDIT",
    "TOTAL_OUTSTANDING_AMOUNT", "TOTAL_OUTSTANDING_FEE",
    "NB_SOS",
]

#convertit les colonnes numériques en type numérique, en remplaçant les valeurs non convertibles par NaN.
_df_num = df_raw[NUMERIC_RAW].apply(pd.to_numeric, errors="coerce")

#calcule les statistiques descriptives pour les colonnes numériques, y compris la moyenne, la médiane, l’écart-type, le minimum, le 95e percentile, le maximum, l’asymétrie (skew), la kurtosis et le pourcentage de valeurs nulles.

stats = pd.DataFrame({
    "mean":    _df_num.mean(), #moyenne (sensible aux valeurs extrêmes.)
    "median":  _df_num.median(),#la valeur centrale de la distribution ; plus fiable que la moyenne quand il y a des valeurs extrêmes. 
    "std":     _df_num.std(), #écart-type (mesure de la dispersion des données autour de la moyenne). si std est élevé, les valeurs sont très éloignées les unes des autres.
    "min":     _df_num.min(),#Elle permet de repérer :des zéros ;des valeurs négatives ;des valeurs incohérentes.
    "p95":     _df_num.quantile(0.95),#95 % des clients ont une valeur <= à cette valeur très utile pour comparer avec le maximum et detecter les valeurs extremes.
    "max":     _df_num.max(),#sert à repérer les très grandes valeurs.
    "skew":    _df_num.skew(),#mesure l’asymétrie de la distribution. Une valeur de skew proche de 0 indique une distribution symétrique, tandis qu’une valeur positive ou négative indique une distribution asymétrique.
    "kurt":    _df_num.kurt(),#mesure la présence de valeurs extrêmes.
    "pct_zero": ((_df_num == 0).sum() / len(_df_num) * 100),#pourcentage de valeurs nulles ou égales à zéro, ce qui peut indiquer une forte concentration de clients sans activité ou avec des montants nuls.+ variable avec beaucoup de zéros peut créer un gros groupe artificiel.
}).round(2)
stats

Les résultats montrent une forte dispersion pour certaines variables, notamment
AVG_CREDIT_AMOUNT, AVG_REIMBURSED_AMOUNT et NB_SOS, dont les valeurs maximales sont
largement supérieures aux médianes. Cela indique la présence de clients ayant des
comportements plus élevés que la majorité. Les coefficients de skewness confirment cette
asymétrie, en particulier pour les variables liées au crédit, au remboursement et à l’usage du
service SOS . De plus , TOTAL_OUTSTANDING_AMOUNT et TOTAL_OUTSTANDING_FEE contiennent
un grand nombre de valeurs nulles, ce qui signifie qu’une partie des clients ne présente aucune
dette restante. Cette analyse met donc en évidence des asymétries, des valeurs extrêmes et
des valeurs nulles qui doivent être prises en compte avant l’application des algorithmes de
clustering.

In [7]:
fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, col in zip(axes.ravel(), NUMERIC_RAW):
    vals = _df_num[col].dropna()
    ax.hist(vals, bins=60, color="steelblue", edgecolor="white")
    ax.set_yscale("log")#l’axe vertical est en échelle logarithmique 
    #Parce que certaines valeurs sont très fréquentes, alors que d’autres sont rares.
    #Sans échelle logarithmique, les petites barres seraient presque invisibles.
    ax.set_title(f"{col}\nskew={vals.skew():.1f}  %0={(vals == 0).mean()*100:.0f}%", fontsize=9)
plt.suptitle("Distributions (y en log) — repérer skewness et point masse à 0", y=1.02)
plt.tight_layout(); plt.show()

### Analyse des distributions des variables numériques

Ce graphique montre que les variables numériques ne suivent pas des distributions équilibrées. Plusieurs variables sont fortement asymétriques, avec une concentration importante de petites valeurs et la présence de quelques valeurs très élevées.

Cette situation peut perturber les algorithmes de clustering basés sur les distances, comme KMeans. En effet, les valeurs extrêmes peuvent influencer fortement le calcul des distances et tirer les centres des clusters vers elles, ce qui peut produire une segmentation moins représentative du comportement général des clients.

C’est pour cette raison que des traitements de préparation ont été appliqués avant la modélisation.

Une échelle logarithmique a également été utilisée sur l’axe vertical (y) afin de mieux visualiser les valeurs rares et les longues traînes de distribution. Cette représentation permet de mieux repérer les valeurs extrêmes et de confirmer la nécessité d’un prétraitement adapté avant le clustering.

**Les longues traînes** montrent qu’il existe quelques clients avec des comportements beaucoup plus élevés que la majorité, par exemple des montants très importants ou un nombre d’utilisations SOS très élevé. Ces cas peuvent influencer le clustering s’ils ne sont pas traités.


### 3.4 Corrélations

On cherche les paires de features **redondantes** (corrélation > 0.9). Elles biaisent la distance euclidienne de KMeans : 2 features colinéaires comptent "2 fois" dans la distance.

In [8]:
corr = _df_num.corr().abs()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=0, vmax=1)
ax.set_xticks(range(len(NUMERIC_RAW))); ax.set_xticklabels(NUMERIC_RAW, rotation=90, fontsize=8)
ax.set_yticks(range(len(NUMERIC_RAW))); ax.set_yticklabels(NUMERIC_RAW, fontsize=8)
plt.colorbar(im, ax=ax, label="|corr|")
ax.set_title("Matrice des corrélations absolues")
plt.tight_layout(); plt.show()

pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(NUMERIC_RAW) for b in NUMERIC_RAW[i+1:] if corr.loc[a, b] > 0.9]
print("Paires fortement corrélées (|corr| > 0.9) :")
for a, b, v in sorted(pairs, key=lambda x: -x[2]):
    print(f"  {a:30s} <-> {b:30s}  = {v:.3f}")

Paires fortement corrélées (|corr| > 0.9) :
  AVG_CREDIT_AMOUNT              <-> AVG_CREDIT_FEE                  = 1.000
  TOTAL_OUTSTANDING_AMOUNT       <-> TOTAL_OUTSTANDING_FEE           = 0.970
  AVG_CREDIT_AMOUNT              <-> AVG_REIMBURSED_AMOUNT           = 0.944
  AVG_CREDIT_FEE                 <-> AVG_REIMBURSED_AMOUNT           = 0.944
  AVG_REIMBURSED_AMOUNT          <-> AVG_FEE_REIMBURSED              = 0.932


### Analyse des corrélations

On observe plusieurs corrélations fortes, notamment entre `AVG_CREDIT_AMOUNT` et `AVG_CREDIT_FEE`, ainsi qu’entre `TOTAL_OUTSTANDING_AMOUNT` et `TOTAL_OUTSTANDING_FEE`.

Cette relation est cohérente d’un point de vue métier, car les frais sont généralement liés aux montants crédités ou aux montants restants dus. Ainsi, certaines variables décrivent des dimensions proches du comportement client et peuvent porter une information redondante.

Cette étape est importante, car KMeans repose sur le calcul de la distance euclidienne. Si deux variables fortement corrélées sont conservées ensemble, une même information peut être prise en compte plusieurs fois dans le calcul de distance. Cela peut biaiser la segmentation en donnant un poids excessif à une seule dimension du comportement client, par exemple les montants financiers.

L’analyse des corrélations permet donc de guider la sélection des variables, de réduire la redondance et, si nécessaire, de construire des ratios plus représentatifs avant la phase de modélisation.


### 3.5 Variabilité par STATE

Question clé : **les features numériques portent-elles de l'info au-delà de STATE ?**  
Si la moyenne d'une feature est quasi identique entre ACTIVE et SUSPENDED, elle n'aide pas à sub-segmenter les ACTIVE.

In [9]:
_df_eda = df_raw.copy() #On crée une copie du dataset brut Pour éviter de modifier directement df_raw
_df_eda[NUMERIC_RAW] = _df_num #on remplace les colonnes numériques par les versions déjà converties en nombres.

by_state = _df_eda.groupby("STATE_IN")[NUMERIC_RAW].median().T
# 1.  regroupe les clients selon leur état 
#2. calcule la médiane de chaque variable numérique pour chaque état    

by_state["var_inter_state"] = by_state.std(axis=1) / (by_state.mean(axis=1).abs() + 1e-9)
#calcule un score de variation entre les états.
#Est-ce que cette variable change beaucoup entre les différents états ?
#Si var_inter_state est élevée, la variable change beaucoup selon l’état du client.
#Si var_inter_state est faible, la variable est presque stable entre les états.
by_state.sort_values("var_inter_state", ascending=False)

### Variabilité des variables numériques selon `STATE_IN`

Cette analyse permet d’évaluer si les variables numériques portent une information différente selon l’état administratif du client.

Pour cela, les clients sont regroupés par `STATE_IN`, puis la médiane de chaque variable numérique est calculée pour chaque état. La médiane est utilisée car elle est moins sensible aux valeurs extrêmes que la moyenne.

Un indicateur de variabilité inter-états, appelé `var_inter_state`, est ensuite calculé. Il mesure la variation relative d’une variable entre les différents états clients. Plus cet indicateur est élevé, plus la variable varie selon l’état du client et plus elle est considérée comme informative.

Les résultats montrent que les variables liées au remboursement, notamment `AVG_FEE_REIMBURSED` et `AVG_REIMBURSED_AMOUNT`, ainsi que la variable `NB_SOS`, présentent une forte variabilité entre les états. Elles peuvent donc contribuer à mieux caractériser les profils clients.

À l’inverse, certaines variables comme `AVG_DAYS_SINCE_CREDIT` présentent une variabilité plus faible, ce qui indique qu’elles différencient moins fortement les états administratifs.

Cette étape permet donc d’identifier les variables les plus porteuses d’information métier avant la phase de clustering.

### 3.6 Diagnostic de clusterisabilité

Avant le clustering, nous vérifions si les clients possèdent suffisamment de signaux comportementaux pour former des groupes distincts.

Un client est considéré comme **inerte** si `NB_SOS`, `TOTAL_OUTSTANDING_AMOUNT` et `TOTAL_OUTSTANDING_FEE` sont tous égaux à zéro.



In [10]:
#les variables qui représentent les signaux principaux du comportement client
SIGNAL_FEATURES = ["NB_SOS", "TOTAL_OUTSTANDING_AMOUNT", "TOTAL_OUTSTANDING_FEE"]

#ette ligne vérifie, pour chaque client, si les 3 variables sont égales à zéro.
inert_mask = (_df_num[SIGNAL_FEATURES] == 0).all(axis=1)
n_inert = int(inert_mask.sum()) #compte le nombre total de clients inertes.
#n_inert = nombre de clients sans signal comportemental.
n_total = len(_df_num)  #compte tous les clients du dataset.

print(f"Clients totalement inertes (NB_SOS=0 ET OUTSTANDING=0 ET OUTSTANDING_FEE=0) :")
print(f"  {n_inert:,} / {n_total:,}  ({n_inert/n_total:.1%})")
print()

for col in SIGNAL_FEATURES + ["AVG_DAYS_SINCE_CREDIT"]:
    pct_zero = (_df_num[col] == 0).mean() * 100 #combien de clients ont une valeur égale à 0 ;
    pct_modal = (_df_num[col] == _df_num[col].mode().iloc[0]).mean() * 100 #combien de clients ont la valeur la plus fréquente.
    #La valeur modale, c’est la valeur qui revient le plus souvent.
    

    print(f"  {col:30s} | % = 0 : {pct_zero:5.1f}% | % à la valeur modale : {pct_modal:5.1f}%")

Clients totalement inertes (NB_SOS=0 ET OUTSTANDING=0 ET OUTSTANDING_FEE=0) :
  0 / 9,748  (0.0%)

  NB_SOS                         | % = 0 :   0.0% | % à la valeur modale :  16.0%
  TOTAL_OUTSTANDING_AMOUNT       | % = 0 :  29.5% | % à la valeur modale :  29.5%
  TOTAL_OUTSTANDING_FEE          | % = 0 :  27.8% | % à la valeur modale :  27.8%
  AVG_DAYS_SINCE_CREDIT          | % = 0 :   0.0% | % à la valeur modale :   0.6%


Le résultat montre qu’aucun client n’est totalement inerte :

```text
0 / 9,748 clients (0.0 %)
```

Le clustering est donc pertinent. Cependant, certaines variables présentent une concentration importante de zéros(zero inflation), notamment `TOTAL_OUTSTANDING_AMOUNT` avec `29.5 %` et `TOTAL_OUTSTANDING_FEE` avec `27.8 %` indiquant qu’une partie importante des clients n’a ni dette restante ni frais restants. Ce diagnostic confirme que le clustering est possible, mais nécessite une préparation adaptée, notamment la création de variables comme  `has_debt`, afin de mieux distinguer les clients avec dette des clients sans dette.



### 3.7 Synthèse EDA — Décisions de prépa

De ce diagnostic, on tire **les règles de conception** du pipeline :

| Problème observé | Conséquence si ignoré | Décision |
|---|---|---|
| Skewness élevée sur OUTSTANDING, NB_SOS | ces valeurs extrêmes peuvent dominer la distance euclidienne utilisée par KMeans| **Winsoriser au p99** puis **Yeo-Johnson** |
| Point masse à 0 très dense | KMeans colle tous les 0 dans un cluster écrasant | **Feature engineering** : créer des flags binaires (`has_debt`, `uses_sos`) pour séparer le point 0 du reste |
| Features OUTSTANDING_AMOUNT ≈ OUTSTANDING_FEE (colinéaires) | Redondance → distance biaisée | **Garder seulement OUTSTANDING_AMOUNT** et construire un `fee_ratio` |
| AVG_REIMBURSED_AMOUNT redondant avec RATIO | idem | **Garder RATIO**, dropper les absolus(Le ratio de remboursement est plus parlant que le montant remboursé seul, car il montre la proportion réellement remboursée.) |
| Clients non-ACTIVE (~5% du total) | Leur STATE suffit à les segmenter | **Clusteriser uniquement les ACTIVE** le but est de segmenter les clients actifs selon leur comportement.|
| Inertes (tout à zéro) | Impossibles à distinguer par clustering | Flag `is_dormant_like` pour les **exclure de la distance** et les suivre à part Sans flag :KMeans découvre tout seul un gros cluster de 0,0,0.Ce cluster peut écraser les autres.Avec flag :On repère ces clients avant.On les met à part.KMeans se concentre sur les vrais comportements.Dans notre dataset, il n’y en a aucun, donc le clustering reste possible. Le flag est seulement prévu par sécurité pour les prochaines données.|

### Nettoyage structurel

Cette étape prépare le dataset avant la modélisation.

Les traitements appliqués sont :

- normalisation de `STATE_IN` afin d’unifier les écritures équivalentes ;
- conversion des variables numériques avec `pd.to_numeric` ;
- conversion des dates avec `pd.to_datetime` ;
- imputation des valeurs manquantes numériques par la médiane ;
- suppression des doublons selon `MSISDN` afin d’avoir une seule ligne par client.

Ce nettoyage garantit une base cohérente, exploitable et adaptée à la phase de clustering.

In [11]:
#Normalisation de STATE_IN
STATE_MAPPING = {
    "ACTIVE": "ACTIVE",
    "SUSPENDED": "SUSPENDED",
    "ON-HOLD": "ON-HOLD",
    "ON HOLD": "ON-HOLD",
    "DISCONNECTED": "DISCONNECTED",
    "DISCONNECTED NETWORK": "DISCONNECTED",
}

#Créer une copie du dataset
df = df_raw.copy()

#Créer une colonne STATE propre
df["STATE"] = (
    df["STATE_IN"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map(STATE_MAPPING)
    .fillna("OTHER")
)

#Conversion des variables numériques en type numérique, en remplaçant les valeurs non convertibles par NaN.
for col in NUMERIC_RAW:
    df[col] = pd.to_numeric(df[col], errors="coerce")

#Conversion des dates en type datetime, en remplaçant les valeurs non convertibles par NaT (Not a Time).
if "ACCOUNT_ACTIVATED_DATE" in df.columns:
    df["ACTIVATION_DATE"] = pd.to_datetime(
        df["ACCOUNT_ACTIVATED_DATE"],
        errors="coerce"
    )
if "LAST_CREDIT_DATE" in df.columns:
    df["LAST_CREDIT_DATE"] = pd.to_datetime(
        df["LAST_CREDIT_DATE"],
        errors="coerce"
    )

#Compter les valeurs manquantes avant imputation
miss_before = df[NUMERIC_RAW].isna().sum().sum() #compte combien de valeurs numériques sont manquantes avant correction.
#Imputation par médiane : pour chaque colonne numérique, on remplace les valeurs manquantes par la médiane de cette colonne.
#variables financières ont des valeurs extrêmes.
#La moyenne peut être faussée par les très grandes valeurs.
#Donc la médiane est plus robuste.
for col in NUMERIC_RAW:
    df[col] = df[col].fillna(df[col].median())

#Suppression des doublons MSISDN
before = len(df)
df = df.drop_duplicates(subset="MSISDN", keep="last").reset_index(drop=True)
#Si un client apparaît plusieurs fois, on garde la dernière occurrence.
dups_removed = before - len(df) #on calcule combien de doublons ont été supprimés.
 
print(f"Manquants imputés (médiane) : {miss_before:,}")
print(f"Doublons MSISDN supprimés   : {dups_removed:,}")
print(f"Shape après nettoyage       : {df.shape}")
print(f"\nRépartition STATE :")
print(df["STATE"].value_counts())

Manquants imputés (médiane) : 0
Doublons MSISDN supprimés   : 0
Shape après nettoyage       : (9748, 16)

Répartition STATE :
STATE
ACTIVE          9208
SUSPENDED        497
DISCONNECTED      28
ON-HOLD           15
Name: count, dtype: int64


### 4.1 Winsorisation au 99e percentile

On applique le plafond **uniquement aux features long-tail** (celles avec `skew > 3` ou `kurt > 10` d'après la section 3.3). Les ratios (`AVG_REIMBURSE_RATIO`) ne sont pas winsorisés : bornés par nature entre 0 et ~1.

In [12]:
#les colonnes à plafonner
WINSORIZE_COLS = [
    "AVG_CREDIT_AMOUNT", "AVG_CREDIT_FEE",
    "AVG_REIMBURSED_AMOUNT", "AVG_FEE_REIMBURSED",
    "AVG_DAYS_SINCE_CREDIT",
    "TOTAL_OUTSTANDING_AMOUNT", "TOTAL_OUTSTANDING_FEE",
    "NB_SOS",
]

winsor_log = []
for col in WINSORIZE_COLS:
    cap = df[col].quantile(0.99) #calcule le 99e percentile de la colonne, c’est-à-dire la valeur en dessous de laquelle se trouvent 99 % des données.
    n_capped = int((df[col] > cap).sum())  #combien de valeurs dépassaient ce plafond.
    df[col] = df[col].clip(upper=cap)  #la winsorisation est appliquée.
    winsor_log.append({"feature": col, "p99_cap": round(cap, 2), "n_capped": n_capped})
#les clients extrêmes peuvent être intéressants On limite juste leur influence dans KMeans.
pd.DataFrame(winsor_log)

## 5. Feature engineering

Après l’EDA, nous avons constaté que certaines variables brutes étaient redondantes ou concentrées autour de zéro.

Pour améliorer le clustering, nous avons créé de nouvelles variables métier :

* **Ratios** : pour résumer les montants et mieux mesurer le remboursement, les frais et la dette.
* **Flags binaires** : pour distinguer la présence ou l’absence d’un comportement, comme avoir une dette ou rembourser presque totalement.
* **Variables temporelles** : pour mesurer l’ancienneté du client et l’intensité d’utilisation du service SOS.

Ces nouvelles variables rendent les profils clients plus faciles à comparer et permettent d’obtenir des clusters plus interprétables.


In [13]:
""""Avant :
montants bruts, frais, dettes, remboursements

Problème :
ça se répète, il y a des zéros, et ce n’est pas toujours clair

Après :
ratios + flags + ancienneté

But :
aider KMeans à créer des groupes plus logiques et plus métier"""

EPS = 1e-6 #une petite valeur pour éviter la division par zéro dans les calculs suivants.

df["reimburse_ratio"]  = df["AVG_REIMBURSE_RATIO"].clip(lower=0, upper=1.5)
#Si la valeur est inférieure à 0, on met 0.
#Si elle dépasse 1.5, on la limite à 1.5.

#Les frais représentent quelle part du montant crédité ?
df["fee_to_credit"]    = df["AVG_CREDIT_FEE"] / (df["AVG_CREDIT_AMOUNT"] + EPS)

#Le montant restant dû est grand ou petit par rapport au crédit moyen ?
df["debt_to_credit"]   = df["TOTAL_OUTSTANDING_AMOUNT"] / (df["AVG_CREDIT_AMOUNT"] + EPS)

#Les frais restants pèsent combien par rapport à la dette restante ?
df["fee_burden"]       = df["TOTAL_OUTSTANDING_FEE"]    / (df["TOTAL_OUTSTANDING_AMOUNT"] + EPS)


#Plafonner les ratios extrêmes
for c in ["fee_to_credit", "debt_to_credit", "fee_burden"]:
    df[c] = df[c].clip(upper=df[c].quantile(0.99))

#--------------Les flags binaires------

#1 = le client a une dette restante /0 = le client n’a pas de dette restante 
df["has_debt"]       = (df["TOTAL_OUTSTANDING_AMOUNT"] > 0).astype(int)

#1 = le client utilise SOS  /  0 = le client n’utilise pas SOS
df["uses_sos"]       = (df["NB_SOS"] > 0).astype(int)

#1 = le client rembourse presque jamais / 0  = sinon
#si le client rembourse moins de 5 %, on le considère comme presque non-rembourseur-->>> regle metier
df["never_repaid"]   = (df["AVG_REIMBURSE_RATIO"] < 0.05).astype(int)
#meme chose 95% ou plus, on considère que le client rembourse toujours
df["full_repayer"]   = (df["AVG_REIMBURSE_RATIO"] >= 0.95).astype(int)


#Ancienneté du client
if "ACTIVATION_DATE" in df.columns:
    ref = df["ACTIVATION_DATE"].max() #ref est la date d’activation la plus récente dans la base.
    df["tenure_days"] = (ref - df["ACTIVATION_DATE"]).dt.days.fillna(0).clip(lower=0) #tenure_days est le nombre de jours depuis l’activation du client jusqu’à la date de référence. Si la date d’activation est manquante, on met 0. Si la date d’activation est dans le futur par rapport à la date de référence, on met aussi 0.
else:
    df["tenure_days"] = 0 #si on n’a pas la date d’activation, on considère que le client n’a pas d’ancienneté.

#Intensité d’utilisation
df["credit_intensity"] = df["NB_SOS"] / (df["tenure_days"] / 30 + 1) #Combien le client utilise SOS par rapport à son ancienneté.
df["credit_intensity"] = df["credit_intensity"].clip(upper=df["credit_intensity"].quantile(0.99)) #On limite aussi les valeurs extrêmes de l’intensité.

#Client dormant / inerte
df["is_dormant_like"] = ( 
    (df["NB_SOS"] == 0) &
    (df["TOTAL_OUTSTANDING_AMOUNT"] == 0) &
    (df["AVG_DAYS_SINCE_CREDIT"] > df["AVG_DAYS_SINCE_CREDIT"].quantile(0.80))
).astype(int)#On utilise le 80e percentile pour repérer les 20 % de clients les plus éloignés de leur dernier crédit.

NEW_FEATURES = [
    "reimburse_ratio", "fee_to_credit", "debt_to_credit", "fee_burden",
    "has_debt", "uses_sos", "never_repaid", "full_repayer",
    "tenure_days", "credit_intensity", "is_dormant_like",
]
df[NEW_FEATURES].describe().T.round(3) #affiche les statistiques des nouvelles variables.

### 5.1 Validation des flags

Après la création des variables binaires, leur répartition est vérifiée afin de s’assurer qu’elles apportent de l’information.

Pour chaque flag, le pourcentage de clients ayant la valeur `1` est calculé sur l’ensemble du dataset, puis uniquement sur les clients `ACTIVE`.

Cette vérification permet d’identifier les variables constantes ou peu informatives. Un flag toujours égal à `0` ou toujours égal à `1` ne permet pas de différencier les clients et apporte donc peu d’intérêt pour le clustering.

In [14]:
FLAGS = ["has_debt", "uses_sos", "never_repaid", "full_repayer", "is_dormant_like"]
flag_summary = pd.DataFrame({
    "% clients = 1 (global)": (df[FLAGS].mean() * 100).round(1),
    "% clients = 1 (ACTIVE)": (df.loc[df.STATE == "ACTIVE", FLAGS].mean() * 100).round(1),
})
flag_summary

## 6. Périmètre ACTIVE + matrice `X` de clustering

### Pourquoi restreindre aux ACTIVE ?

- **SUSPENDED / ON-HOLD / DISCONNECTED** sont déjà **segmentés** par leur état administratif — c'est une catégorisation imposée, pas à découvrir.
- Les inclure force l'algorithme à "retrouver" STATE comme axe principal → on perd la finesse sur les ACTIVE, qui sont **95% du parc** et **les seuls exploitables**

**Architecture finale** qui en découle :

```
Tout le parc
├── STATE ≠ ACTIVE → segment = STATE (déterministe, pas de ML)
└── STATE = ACTIVE  → clustering ML → cluster_name
```

### Sélection des features pour `X`

On **exclut** :
- Les features brutes redondantes (`AVG_REIMBURSED_AMOUNT`, `AVG_FEE_REIMBURSED`) — remplacées par les ratios.
- `TOTAL_OUTSTANDING_FEE` — fortement corrélée à `TOTAL_OUTSTANDING_AMOUNT`.
- `is_dormant_like` — servira étiquetage, pas en distance.

In [15]:
#garde seulement les clients actifs
df_active = df[df["STATE"] == "ACTIVE"].copy().reset_index(drop=True)
print(f"Clients ACTIVE : {len(df_active):,}  ({len(df_active)/len(df):.1%} du parc)")

CLUSTER_FEATURES = [
    "AVG_CREDIT_AMOUNT",
    "reimburse_ratio", #AVG_REIMBURSED_AMOUNT, AVG_FEE_REIMBURSED
    "AVG_DAYS_SINCE_CREDIT",
    "TOTAL_OUTSTANDING_AMOUNT", #TOTAL_OUTSTANDING_FEE
    "NB_SOS",
    "fee_to_credit",
    "debt_to_credit",
    "fee_burden",#TOTAL_OUTSTANDING_FEE
    "credit_intensity",
    "tenure_days",
    "has_debt",
    "uses_sos",
    "never_repaid",
    "full_repayer",
]

print(f"\nFeatures retenues pour X ({len(CLUSTER_FEATURES)}) :")
for f in CLUSTER_FEATURES:
    print(f"  - {f}")
#Cette ligne transforme le tableau pandas en matrice numérique.
#Parce que les modèles scikit-learn comme KMeans travaillent avec des matrices numériques.
X_raw = df_active[CLUSTER_FEATURES].to_numpy(dtype=float)
print(f"\nShape X : {X_raw.shape}")

Clients ACTIVE : 9,208  (94.5% du parc)

Features retenues pour X (14) :
  - AVG_CREDIT_AMOUNT
  - reimburse_ratio
  - AVG_DAYS_SINCE_CREDIT
  - TOTAL_OUTSTANDING_AMOUNT
  - NB_SOS
  - fee_to_credit
  - debt_to_credit
  - fee_burden
  - credit_intensity
  - tenure_days
  - has_debt
  - uses_sos
  - never_repaid
  - full_repayer

Shape X : (9208, 14)


### 6.1 Transformation finale : Yeo-Johnson puis RobustScaler

**Pourquoi Yeo-Johnson** ?

- Il accepte les **valeurs nulles et négatives** (contrairement à `log1p` qui échoue sur négatifs).
- Il **apprend** le paramètre de puissance optimal par feature pour se rapprocher d'une gaussienne.
- Il est **monotone** → préserve l'ordre des clients.

Après Yeo-Johnson, `RobustScaler` centre sur la médiane et normalise par l'IQR : insensible aux résidus d'outliers.

**Important** : on `fit` sur un **échantillon de 200k** pour la vitesse, puis on `transform` tout le dataset.

In [16]:


FIT_SAMPLE = 200_000

rng = np.random.default_rng(RANDOM_STATE)
fit_idx = rng.choice(len(X_raw), size=min(FIT_SAMPLE, len(X_raw)), replace=False)
# Yeo-Johnson apprend comment transformer chaque variable
# Objectif : réduire l'asymétrie des distributions
# standardize=False car la normalisation sera faite ensuite avec RobustScaler
yj = PowerTransformer(method="yeo-johnson", standardize=False).fit(X_raw[fit_idx])
# Application de la transformation Yeo-Johnson sur toute la matrice X_raw
X_yj = yj.transform(X_raw)
# RobustScaler apprend la médiane et l'IQR sur l'échantillon
# Objectif : mettre les variables sur une échelle comparable
# Il est robuste aux valeurs extrêmes, contrairement à StandardScaler qui utilise la moyenne et l'écart-type.
rs = RobustScaler().fit(X_yj[fit_idx])

# Application de la normalisation sur toute la matrice transformée
# X devient la matrice finale prête pour le clustering
X = rs.transform(X_yj)

# Calcul de l'écart-type de chaque feature
# Si std = 0 ou presque 0, la variable ne varie pas
std_per_feature = X.std(axis=0)
# On garde uniquement les features qui ont une vraie variation
# 1e-4 est un petit seuil pour détecter les variables quasi constantes
keep_mask = std_per_feature > 1e-4
dropped = [CLUSTER_FEATURES[i] for i, keep in enumerate(keep_mask) if not keep]

# Liste des features supprimées car elles n'apportent pas d'information
if dropped:
    print(f"Features à variance nulle retirées : {dropped}")
    CLUSTER_FEATURES = [f for i, f in enumerate(CLUSTER_FEATURES) if keep_mask[i]]
    X_raw = X_raw[:, keep_mask]
    X = X[:, keep_mask]

print(f"X shape : {X.shape}")
print(f"Features actives ({len(CLUSTER_FEATURES)}): {CLUSTER_FEATURES}")
print(f"std     : {X.std(axis=0).round(3)}")

# Création d'une figure avec 2 lignes et 4 colonnes
# Ligne 1 : données brutes winsorisées
# Ligne 2 : données après Yeo-Johnson + RobustScaler
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
n_show = min(30_000, len(X))
idx_show = rng.choice(len(X), size=n_show, replace=False)
cols_show = [CLUSTER_FEATURES.index(c) for c in ["NB_SOS", "TOTAL_OUTSTANDING_AMOUNT",
              "AVG_CREDIT_AMOUNT", "reimburse_ratio"]]
for i, (col_idx, col_name) in enumerate(zip(cols_show, ["NB_SOS", "OUTSTANDING", "CREDIT_AMT", "RATIO"])):
    axes[0, i].hist(X_raw[idx_show, col_idx], bins=60, color="crimson", edgecolor="white")
    axes[0, i].set_title(f"{col_name} — brut (winsorisé)", fontsize=9)
    axes[0, i].set_yscale("log")
    axes[1, i].hist(X[idx_show, col_idx], bins=60, color="seagreen", edgecolor="white")
    axes[1, i].set_title(f"{col_name} — Yeo-Johnson + Robust", fontsize=9)
plt.tight_layout(); plt.show()

Features à variance nulle retirées : ['fee_to_credit', 'uses_sos']
X shape : (9208, 12)
Features actives (12): ['AVG_CREDIT_AMOUNT', 'reimburse_ratio', 'AVG_DAYS_SINCE_CREDIT', 'TOTAL_OUTSTANDING_AMOUNT', 'NB_SOS', 'debt_to_credit', 'fee_burden', 'credit_intensity', 'tenure_days', 'has_debt', 'never_repaid', 'full_repayer']
std     : [0.62  0.517 0.863 0.504 0.645 0.485 0.624 0.586 0.645 0.458 0.005 0.476]


In [17]:
zero_var_cols = [CLUSTER_FEATURES[i] for i, s in enumerate(X.std(axis=0)) if s < 1e-4]
print("Features à variance nulle :", zero_var_cols)

Features à variance nulle : []


## 7. Choix de K — 4 métriques croisées

Un seul indicateur peut mentir. On croise :

| Métrique | Lecture | Faiblesse seule |
|---|---|---|
| **Inertie** (elbow) | Coude visuel | Subjective |
| **Silhouette** | [-1, 1], plus haut = mieux | Peut être élevée artificiellement si clusters très déséquilibrés |
| **Davies-Bouldin** | Plus bas = mieux | Biaisée vers clusters sphériques |
| **Calinski-Harabasz** | Plus haut = mieux | Favorise les grands K |

**Contrainte métier** qu'on ajoute : **aucun cluster ne doit représenter < 0.5% des clients** (sinon inexploitable par les agents).

Calcul sur un **sous-ensemble déterministe plafonné des clients ACTIVE** pour maintenir un temps d'exécution raisonnable, tout en conservant une approche reproductible.

In [18]:
# On fixe le nombre maximum de clients utilisés pour évaluer K.
# Ici, on prend au maximum 3000 clients pour accélérer les calculs.
K_EVAL_SAMPLE_SIZE = min(3000, len(X))

# On sélectionne aléatoirement des indices de clients dans X.
# Comme rng est basé sur RANDOM_STATE, le tirage reste reproductible.
k_idx = rng.choice(
    len(X), 
    size=K_EVAL_SAMPLE_SIZE, 
    replace=False
)

# On crée un sous-ensemble de la matrice X.
# Ce sous-ensemble servira uniquement à comparer les valeurs de K.
X_k = X[k_idx]

# Nombre de lignes utilisées pour calculer la silhouette.
# Ici, maximum 3000 lignes.
sil_sample = min(3000, len(X_k))

# On définit les valeurs de K à tester.
# range(2, 9) veut dire : 2, 3, 4, 5, 6, 7, 8.
K_RANGE = range(2, 9)

# Liste vide qui va stocker les résultats pour chaque valeur de K.
k_results = []

# On teste chaque valeur de K une par une.
for k in K_RANGE:

    # Création du modèle MiniBatchKMeans pour le K actuel.
    # n_clusters=k : nombre de clusters à créer.
    # random_state : garantit des résultats reproductibles.
    # batch_size : taille des lots utilisés pour accélérer KMeans.
    # n_init=5 : le modèle est lancé 5 fois et garde la meilleure solution.
    # max_iter=200 : nombre maximum d'itérations.
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        batch_size=min(2048, len(X_k)),
        n_init=5,
        max_iter=200,
    )

    # Le modèle apprend les clusters et attribue un cluster à chaque client.
    labels = km.fit_predict(X_k)

    # On compte le nombre de clients dans chaque cluster.
    sizes = np.bincount(labels, minlength=k)

    # On calcule le pourcentage du plus petit cluster.
    # Cela permet de vérifier s'il existe un cluster trop petit.
    min_pct = sizes.min() / len(labels) * 100

    # Calcul du score de silhouette.
    # Plus il est élevé, meilleure est la séparation des clusters.
    sil = silhouette_score(
        X_k, 
        labels, 
        sample_size=sil_sample, 
        random_state=RANDOM_STATE
    )

    # Calcul du score Davies-Bouldin.
    # Plus il est faible, meilleurs sont les clusters.
    db = davies_bouldin_score(X_k, labels)

    # Calcul du score Calinski-Harabasz.
    # Plus il est élevé, meilleure est la séparation globale.
    ch = calinski_harabasz_score(X_k, labels)

    # On stocke tous les résultats de cette valeur de K dans la liste.
    k_results.append({
        "K": k,                              # valeur de K testée
        "inertie": km.inertia_,             # compacité des clusters
        "silhouette": sil,                  # qualité de séparation
        "davies_bouldin": db,               # qualité compacité/séparation
        "calinski_harabasz": ch,            # séparation globale
        "min_cluster_pct": round(min_pct, 2), # taille du plus petit cluster
        "viable_metier": "OK" if min_pct >= 0.5 else "TROP PETIT",
    })

# On transforme la liste des résultats en DataFrame pour afficher un tableau.
k_df = pd.DataFrame(k_results)

# On affiche le tableau avec 3 chiffres après la virgule.
k_df.round(3)

In [19]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
metrics_cfg = [
    ("inertie",           "Inertie (↓ = mieux, chercher le coude)"),
    ("silhouette",        "Silhouette (↑ = mieux)"),
    ("davies_bouldin",    "Davies-Bouldin (↓ = mieux)"),
    ("calinski_harabasz", "Calinski-Harabasz (↑ = mieux)"),
]
for ax, (m, title) in zip(axes.ravel(), metrics_cfg):
    ax.plot(k_df["K"], k_df[m], marker="o", color="steelblue")
    ax.set_xlabel("K"); ax.set_title(title, fontsize=10); ax.grid(alpha=0.3)
plt.suptitle("Métriques d'évaluation en fonction de K", y=1.02)
plt.tight_layout(); plt.show()

viable = k_df[k_df["viable_metier"] == "OK"]
if len(viable):
    k_best = int(viable.sort_values("silhouette", ascending=False).iloc[0]["K"])
else:
    k_best = int(k_df.sort_values("silhouette", ascending=False).iloc[0]["K"])
print(f"K retenu : {k_best}  (meilleure silhouette avec min_cluster_pct >= 0.5%)")

K retenu : 2  (meilleure silhouette avec min_cluster_pct >= 0.5%)


## 8. Comparaison de plusieurs algorithmes de clustering

Après avoir déterminé le nombre optimal de clusters, fixé à `K = 2`, nous comparons plusieurs algorithmes de clustering afin de sélectionner le modèle le plus adapté.

L’objectif est de ne pas retenir un algorithme uniquement par défaut, mais de comparer plusieurs approches avec le même nombre de clusters et les mêmes données d’entrée.

Trois algorithmes sont évalués :

| Algorithme          | Principe                                                                  | Intérêt dans ce projet                                                    |
| ------------------- | ------------------------------------------------------------------------- | ------------------------------------------------------------------------- |
| **MiniBatchKMeans** | Variante rapide de KMeans, basée sur la proximité aux centres de clusters | Méthode simple, rapide, interprétable et adaptée à des volumes importants |
| **BisectingKMeans** | Découpe progressivement les données en sous-groupes                       | Utile si les groupes de clients ont des tailles différentes               |
| **GaussianMixture** | Modèle probabiliste permettant une appartenance plus souple aux clusters  | Intéressant si les profils clients se chevauchent                         |

Chaque modèle est entraîné sur un échantillon représentatif des clients `ACTIVE`, puis appliqué à l’ensemble des clients actifs. Les algorithmes sont ensuite comparés à l’aide des mêmes métriques : silhouette, Davies-Bouldin, Calinski-Harabasz et répartition des tailles de clusters.

Cette comparaison permet de choisir le modèle offrant le meilleur compromis entre qualité statistique, stabilité et exploitabilité métier.


In [20]:
# Dictionnaire contenant les 3 algorithmes de clustering à comparer
# Tous utilisent le même nombre de clusters : k_best
algos = {

    # MiniBatchKMeans : version rapide de KMeans
    # Il regroupe les clients autour de centres de clusters
    "MiniBatchKMeans": MiniBatchKMeans(
        n_clusters=k_best,                     # nombre de clusters retenu précédemment
        random_state=RANDOM_STATE,             # rend les résultats reproductibles
        batch_size=min(4096, len(X_k)),        # taille des mini-lots utilisés pour accélérer l'entraînement
        n_init=10,                             # relance l'algorithme 10 fois et garde la meilleure solution
        #plusieurs initialisations pour réduire le risque d’obtenir un mauvais clustering à cause d’un mauvais point de départ.
        max_iter=300                           # nombre maximal d'itérations
        #max_iter fixe une limite au nombre d’itérations afin de laisser au modèle le temps de converger, tout en évitant un calcul infini.
    ),

    # BisectingKMeans : variante qui coupe progressivement les groupes en deux
    # Utile si les clusters peuvent avoir des tailles différentes
    "BisectingKMeans": BisectingKMeans(
        n_clusters=k_best,                     # nombre de clusters à créer
        random_state=RANDOM_STATE,             # reproductibilité
        n_init=3                               # nombre d'initialisations
    ),

    # GaussianMixture : modèle probabiliste
    # Il est plus souple car les clusters peuvent se chevaucher
    "GaussianMixture": GaussianMixture(
        n_components=k_best,                   # équivalent du nombre de clusters
        random_state=RANDOM_STATE,             # reproductibilité
        covariance_type="full",                # forme libre des groupes
        n_init=3,                              # nombre d'initialisations
        max_iter=200                           # nombre maximal d'itérations
    ),
}

# Liste qui va contenir les scores de chaque algorithme
algo_scores = []

# Dictionnaire qui va stocker les labels de clusters prédits pour tous les clients ACTIVE
algo_labels_full = {}

# Boucle pour tester chaque algorithme un par un
for name, model in algos.items():

    # Entraînement du modèle sur l'échantillon X_k
    # X_k est utilisé pour accélérer le calcul
    model.fit(X_k)

    # Application du modèle entraîné à toute la matrice X
    # labs contient le cluster attribué à chaque client ACTIVE
    labs = model.predict(X)

    # Calcul du nombre de clients dans chaque cluster
    sizes = np.bincount(labs, minlength=k_best)

    # Calcul du score de silhouette sur l'échantillon X_k
    # Utilité : mesure si les clusters sont bien séparés
    # Plus la silhouette est élevée, meilleur est le clustering
    sil = silhouette_score(
        X_k,
        model.predict(X_k),
        sample_size=sil_sample,
        random_state=RANDOM_STATE
    )

    # Calcul du score Davies-Bouldin
    # Utilité : mesure si les clusters sont compacts et bien séparés
    # Plus Davies-Bouldin est faible, meilleur est le clustering
    db = davies_bouldin_score(
        X_k,
        model.predict(X_k)
    )

    # Calcul du score Calinski-Harabasz
    # Utilité : compare la séparation entre clusters avec la dispersion interne
    # Plus Calinski-Harabasz est élevé, meilleur est le clustering
    ch = calinski_harabasz_score(
        X_k,
        model.predict(X_k)
    )

    # Stockage des résultats de l'algorithme courant
    algo_scores.append({

        # Nom de l'algorithme testé
        "algo": name,

        # Silhouette : plus haut = mieux
        # Indique si les clients sont bien dans leur cluster et loin des autres clusters
        "silhouette": round(sil, 4),

        # Davies-Bouldin : plus bas = mieux
        # Indique si les clusters sont compacts et bien séparés
        "davies_bouldin": round(db, 4),

        # Calinski-Harabasz : plus haut = mieux
        # Indique si la séparation globale entre les clusters est bonne
        "calinski_harabasz": round(ch, 1),

        # Pourcentage du plus petit cluster
        # Utilité métier : éviter un cluster trop petit et inexploitable
        "min_size_pct": round(sizes.min() / len(labs) * 100, 2),

        # Pourcentage du plus grand cluster
        # Utilité : voir si un cluster domine trop les autres
        "max_size_pct": round(sizes.max() / len(labs) * 100, 2),

        # Rapport entre le plus grand et le plus petit cluster
        # Utilité : mesurer le déséquilibre des tailles
        # Plus le ratio est faible, plus la répartition est équilibrée
        "size_ratio": round(sizes.max() / max(sizes.min(), 1), 1),
    })

    # On sauvegarde les labels produits par cet algorithme
    # Cela permet de récupérer ensuite les clusters du meilleur modèle
    algo_labels_full[name] = labs

# Transformation des scores en tableau comparatif
comparison = pd.DataFrame(algo_scores)

# Affichage du tableau de comparaison
comparison

### 8.1 Sélection de l’algorithme retenu

Après la comparaison des algorithmes, le choix final est fait selon une règle simple :

1. **Éliminer les modèles avec un cluster trop petit**
   Un cluster inférieur à `0,5 %` des clients est considéré comme peu exploitable métier.

2. **Choisir la répartition la moins déséquilibrée**
   On privilégie le modèle avec le `size_ratio` le plus faible, c’est-à-dire celui qui évite un très grand cluster et un très petit cluster.

3. **Utiliser la silhouette en cas d’égalité**
   Si plusieurs modèles ont une répartition proche, on garde celui qui sépare le mieux les clusters.

Le modèle retenu est **MiniBatchKMeans**.

Il donne deux clusters exploitables :

* **Cluster 0** : 6602 clients, soit **71,7 %**
* **Cluster 1** : 2606 clients, soit **28,3 %**

Cette répartition montre que les deux segments sont suffisamment représentatifs pour être utilisés dans la plateforme.


In [21]:
# ------------------------------------------------------------
# Sélection de l'algorithme final parmi les modèles comparés
# ------------------------------------------------------------

# On garde uniquement les algorithmes dont le plus petit cluster
# représente au moins 0,5 % des clients.
# Objectif : éliminer les modèles qui créent des micro-clusters
# difficilement exploitables métier.
viable = comparison[comparison["min_size_pct"] >= 0.5].copy()

# Sécurité :
# si aucun modèle ne respecte la contrainte métier,
# on reprend tous les modèles pour éviter de bloquer le traitement.
if len(viable) == 0:
    viable = comparison.copy()


# ------------------------------------------------------------
# Classement des modèles viables
# ------------------------------------------------------------

# On trie les modèles selon deux critères :
# 1) size_ratio croissant :
#    on préfère le modèle qui donne la répartition la moins déséquilibrée.
#    Un size_ratio faible signifie que le plus grand cluster
#    n'est pas énormément plus grand que le plus petit.
#
# 2) silhouette décroissante :
#    si deux modèles ont une répartition proche,
#    on garde celui qui sépare le mieux les clusters.
viable = viable.sort_values(
    ["size_ratio", "silhouette"],
    ascending=[True, False]
)


# ------------------------------------------------------------
# Récupération du meilleur algorithme
# ------------------------------------------------------------

# Après le tri, le meilleur modèle est placé en première ligne.
# On récupère son nom dans la colonne "algo".
best_algo = viable.iloc[0]["algo"]

# Affichage du modèle retenu.
print(f"Algo retenu : {best_algo}")


# ------------------------------------------------------------
# Récupération des labels du modèle retenu
# ------------------------------------------------------------

# algo_labels_full contient les clusters prédits par chaque algorithme.
# On récupère ici les labels correspondant à l'algorithme final retenu.
labels = algo_labels_full[best_algo]

# On ajoute les clusters dans le dataframe des clients ACTIVE.
# Chaque client actif reçoit donc un cluster_id.
df_active["cluster_id"] = labels


# ------------------------------------------------------------
# Vérification de la répartition finale des clusters
# ------------------------------------------------------------

# On compte le nombre de clients dans chaque cluster.
cluster_sizes = pd.Series(labels).value_counts().sort_index()

# On calcule le pourcentage de clients dans chaque cluster.
cluster_pct = (cluster_sizes / len(labels) * 100).round(2)

# On construit un tableau final avec :
# - count : nombre de clients par cluster
# - pct   : pourcentage de clients par cluster
cluster_distrib = pd.DataFrame({
    "count": cluster_sizes,
    "pct": cluster_pct
})

# Affichage de la répartition finale.
print("\nRépartition finale :")
print(cluster_distrib)

Algo retenu : MiniBatchKMeans

Répartition finale :
   count    pct
0   6602 71.700
1   2606 28.300


## 9. Interprétation et nommage automatique des clusters

Après l’affectation des clients aux clusters, cette étape permet de donner un sens métier aux groupes obtenus.

Pour chaque cluster, on calcule d’abord le profil moyen à partir des variables métier : montant de crédit, remboursement, dette, fréquence d’usage SOS, ancienneté et intensité d’utilisation.

Ensuite, chaque profil de cluster est comparé à la moyenne globale des clients `ACTIVE` à l’aide d’un z-score.
Un z-score positif signifie que le cluster est au-dessus de la moyenne, tandis qu’un z-score négatif signifie qu’il est en dessous.

Les variables dont le z-score dépasse `0,5` en valeur absolue sont considérées comme des traits distinctifs du cluster.

Ces traits sont ensuite traduits en noms métier grâce à des règles simples :

* `NB_SOS` élevé → **SOS-heavy**
* `debt_to_credit` élevé → **Endetté**
* `reimburse_ratio` élevé → **Bon-payeur**
* `reimburse_ratio` faible → **Mauvais-payeur**
* `AVG_DAYS_SINCE_CREDIT` élevé → **Latent**
* `credit_intensity` élevé → **Intensif**

Cette étape permet donc de transformer des numéros de clusters en segments compréhensibles et exploitables par l’agent décisionnel.


In [22]:
# ------------------------------------------------------------
# Interprétation des clusters
# ------------------------------------------------------------

# On choisit les variables métier qui vont servir à comprendre les clusters.
# Ces variables décrivent le comportement des clients :
# crédit, remboursement, dette, usage SOS, ancienneté et intensité.
PROFILE_COLS = [
    "AVG_CREDIT_AMOUNT", "reimburse_ratio", "AVG_DAYS_SINCE_CREDIT",
    "TOTAL_OUTSTANDING_AMOUNT", "NB_SOS", "debt_to_credit",
    "credit_intensity", "tenure_days", "has_debt", "uses_sos",
    "never_repaid", "full_repayer", "is_dormant_like",
]


# ------------------------------------------------------------
# 1) Calcul du profil moyen de chaque cluster
# ------------------------------------------------------------

# On regroupe les clients par cluster_id.
# Puis on calcule la moyenne de chaque variable pour chaque cluster.
# Cela permet de savoir, par exemple, si un cluster utilise plus SOS,
# rembourse mieux ou possède plus de dette.
profile_mean = df_active.groupby("cluster_id")[PROFILE_COLS].mean()


# ------------------------------------------------------------
# 2) Calcul de la moyenne globale des clients ACTIVE
# ------------------------------------------------------------

# On calcule la moyenne de chaque variable sur tous les clients actifs.
# Cette moyenne globale sert de référence pour comparer les clusters.
pop_mean = df_active[PROFILE_COLS].mean()


# ------------------------------------------------------------
# 3) Calcul de l'écart-type global
# ------------------------------------------------------------

# L'écart-type mesure la dispersion des valeurs autour de la moyenne.
# Il sert à standardiser l'écart entre un cluster et la moyenne globale.
#
# replace(0, 1) évite une division par zéro si une variable ne varie pas.
pop_std = df_active[PROFILE_COLS].std().replace(0, 1)


# ------------------------------------------------------------
# 4) Calcul des z-scores
# ------------------------------------------------------------

# Le z-score mesure l'écart entre le profil moyen d'un cluster
# et la moyenne globale des clients ACTIVE.
#
# z-score positif  : le cluster est au-dessus de la moyenne
# z-score négatif  : le cluster est en dessous de la moyenne
# z-score proche 0 : le cluster est proche de la moyenne
#
# Exemple :
# z(NB_SOS) = +0.8  -> le cluster utilise plus SOS que la moyenne
# z(reimburse_ratio) = -0.7 -> le cluster rembourse moins bien que la moyenne
zscores = (profile_mean - pop_mean) / pop_std


# ------------------------------------------------------------
# 5) Affichage du profil moyen des clusters
# ------------------------------------------------------------

print("Profils moyens par cluster :")

# On copie le tableau des moyennes par cluster pour l'affichage.
display_profile = profile_mean.copy()

# On ajoute le nombre de clients dans chaque cluster.
display_profile["size"] = df_active.groupby("cluster_id").size()

# On ajoute le pourcentage de clients dans chaque cluster.
display_profile["pct"] = (
    display_profile["size"] / len(df_active) * 100
).round(2)

# On affiche le tableau avec deux chiffres après la virgule.
display_profile.round(2)

Profils moyens par cluster :


In [23]:
print("Z-scores (écart à la moyenne ACTIVE, en écarts-types) :")
zscores.round(2)

Z-scores (écart à la moyenne ACTIVE, en écarts-types) :


###  Nommage automatique des clusters

Après le calcul des z-scores, chaque cluster est nommé automatiquement à partir de ses caractéristiques les plus distinctives.

Un seuil de `0,5` est utilisé pour identifier les variables qui s’écartent suffisamment de la moyenne des clients `ACTIVE`. Lorsqu’une variable dépasse ce seuil, elle est traduite en tag métier.

Les principales règles utilisées sont les suivantes :

* `NB_SOS` élevé → **SOS-heavy**
* `debt_to_credit` élevé → **Endetté**
* `TOTAL_OUTSTANDING_AMOUNT` élevé → **Encours-élevé**
* `reimburse_ratio` élevé → **Bon-payeur**
* `reimburse_ratio` faible → **Mauvais-payeur**
* `AVG_DAYS_SINCE_CREDIT` élevé → **Latent**
* `credit_intensity` élevé → **Intensif**
* `AVG_CREDIT_AMOUNT` élevé → **Gros-ticket**
* `tenure_days` élevé → **Ancien**
* `is_dormant_like` élevé → **Dormant-like**

Si aucun trait distinctif n’est détecté, le cluster est nommé **Standard**.

Le nom final peut combiner plusieurs tags, par exemple :
**SOS-heavy + Endetté + Mauvais-payeur**.

Cette étape permet de transformer les identifiants numériques des clusters en segments métier compréhensibles et directement exploitables par l’agent décisionnel.


In [24]:
# Seuil utilisé pour décider si une variable est distinctive.
# Si le z-score dépasse +0.5 ou -0.5, la variable est considérée importante.
Z_THRESHOLD = 0.5


# Fonction qui reçoit les z-scores d'un cluster
# et retourne un nom métier basé sur les caractéristiques fortes.
def name_cluster(z_row: pd.Series) -> str:
    
    # Liste qui va contenir les tags du cluster.
    tags = []

    # Si le cluster utilise beaucoup plus SOS que la moyenne.
    if z_row.get("NB_SOS", 0) > Z_THRESHOLD:
        tags.append("SOS-heavy")

    # Si le cluster est plus endetté que la moyenne.
    if z_row.get("debt_to_credit", 0) > Z_THRESHOLD:
        tags.append("Endetté")

    # Si l'encours restant est élevé.
    # On ajoute "Encours-élevé" seulement si "Endetté" n'a pas déjà été ajouté,
    # pour éviter de répéter deux fois la même idée.
    if z_row.get("TOTAL_OUTSTANDING_AMOUNT", 0) > Z_THRESHOLD and "Endetté" not in tags:
        tags.append("Encours-élevé")

    # Si le cluster rembourse mieux que la moyenne.
    if z_row.get("reimburse_ratio", 0) > Z_THRESHOLD:
        tags.append("Bon-payeur")

    # Si le cluster rembourse moins bien que la moyenne.
    if z_row.get("reimburse_ratio", 0) < -Z_THRESHOLD:
        tags.append("Mauvais-payeur")

    # Si le dernier crédit est plus ancien que la moyenne.
    if z_row.get("AVG_DAYS_SINCE_CREDIT", 0) > Z_THRESHOLD:
        tags.append("Latent")

    # Si l'utilisation du service SOS est plus intensive.
    if z_row.get("credit_intensity", 0) > Z_THRESHOLD:
        tags.append("Intensif")

    # Si le montant moyen du crédit est plus élevé.
    if z_row.get("AVG_CREDIT_AMOUNT", 0) > Z_THRESHOLD:
        tags.append("Gros-ticket")

    # Si les clients du cluster sont plus anciens.
    if z_row.get("tenure_days", 0) > Z_THRESHOLD:
        tags.append("Ancien")

    # Si le cluster ressemble à un profil dormant.
    if z_row.get("is_dormant_like", 0) > Z_THRESHOLD:
        tags.append("Dormant-like")

    # Si aucun tag n'a été détecté, le cluster est considéré standard.
    if not tags:
        return "Standard"

    # On combine les tags pour construire le nom du cluster.
    # On garde au maximum les 3 premiers tags pour éviter des noms trop longs.
    return " + ".join(tags[:3])


# Application de la fonction à chaque cluster.
# On crée un dictionnaire : cluster_id -> nom métier.
cluster_names = {
    cid: name_cluster(zscores.loc[cid])
    for cid in zscores.index
}

# Ajout du nom métier dans le dataframe des clients actifs.
df_active["cluster_name"] = df_active["cluster_id"].map(cluster_names)


# Création d'un résumé final :
# nombre de clients par cluster_id et cluster_name.
summary = (
    df_active.groupby(["cluster_id", "cluster_name"])
      .size()
      .reset_index(name="count")
)

# Ajout du pourcentage de clients par segment.
summary["pct"] = (
    summary["count"] / len(df_active) * 100
).round(2)

# Affichage des segments du plus grand au plus petit.
summary.sort_values("count", ascending=False)

L’interprétation des clusters montre une séparation claire entre deux profils principaux :

Cluster 0 : clients endettés avec remboursement partiel
Cluster 1 : clients bons payeurs sans dette

Cette étape permet de transformer les numéros de clusters en segments métier compréhensibles et exploitables par l’agent décisionnel.


### 9.1 Visualisations : heatmap + PCA 2D

In [25]:
# Création d'une figure et d'un axe pour le graphique.
# figsize=(11, 5) définit la taille du graphique : 11 pouces de largeur et 5 pouces de hauteur.
fig, ax = plt.subplots(figsize=(11, 5))

# On transpose le tableau des z-scores.
# Au départ, zscores contient les clusters en lignes et les variables en colonnes.
# Avec .T, on inverse : les variables deviennent les lignes et les clusters deviennent les colonnes.
# .values transforme le DataFrame pandas en tableau numpy pour l'affichage avec imshow.
z_plot = zscores.T.values

# Création de la heatmap.
# imshow affiche les valeurs numériques sous forme de couleurs.
# cmap="RdBu_r" utilise une palette bleu-blanc-rouge inversée :
# - bleu pour les valeurs négatives,
# - blanc pour les valeurs proches de 0,
# - rouge pour les valeurs positives.
# vmin=-2 et vmax=2 fixent les limites de couleur pour faciliter la comparaison.
# aspect="auto" adapte automatiquement la forme des cases.
im = ax.imshow(
    z_plot,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    aspect="auto"
)

# On définit les positions des labels sur l'axe Y.
# Il y a une ligne par variable métier dans PROFILE_COLS.
ax.set_yticks(range(len(PROFILE_COLS)))

# On affiche le nom des variables métier sur l'axe Y.
# fontsize=9 réduit légèrement la taille du texte pour éviter que les noms se chevauchent.
ax.set_yticklabels(PROFILE_COLS, fontsize=9)

# On prépare les labels de l'axe X.
# Chaque colonne correspond à un cluster.
# Exemple : C0 Standard ou C1 Bon-payeur.
# cluster_names contient le nom métier attribué automatiquement à chaque cluster.
x_labels = [
    f"C{cid}\n{cluster_names[cid]}"
    for cid in zscores.index
]

# On définit les positions des labels sur l'axe X.
# Il y a une position par cluster.
ax.set_xticks(range(len(x_labels)))

# On affiche les noms des clusters sur l'axe X.
# fontsize=8 permet de garder un affichage lisible.
ax.set_xticklabels(x_labels, fontsize=8)

# Boucle sur toutes les lignes de la heatmap.
# i représente l'indice de la variable métier.
for i in range(z_plot.shape[0]):

    # Boucle sur toutes les colonnes de la heatmap.
    # j représente l'indice du cluster.
    for j in range(z_plot.shape[1]):

        # On affiche la valeur numérique uniquement si elle est significative.
        # Ici, une valeur est considérée comme significative si son z-score
        # est supérieur ou égal à Z_THRESHOLD en valeur absolue.
        # Par exemple : +0.7 ou -0.8 seront affichés.
        if abs(z_plot[i, j]) >= Z_THRESHOLD:

            # Ajout du texte au centre de la case.
            # f"{z_plot[i, j]:+.1f}" affiche le signe + ou - et garde 1 chiffre après la virgule.
            # ha="center" centre horizontalement le texte.
            # va="center" centre verticalement le texte.
            # Si la valeur est très forte, le texte est en blanc pour rester lisible.
            # Sinon, le texte est en noir.
            ax.text(
                j,
                i,
                f"{z_plot[i, j]:+.1f}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if abs(z_plot[i, j]) > 1.2 else "black"
            )

# Ajout d'une barre de couleur à droite du graphique.
# Elle permet de comprendre la signification des couleurs.
plt.colorbar(im, ax=ax, label="z-score")

# Ajout du titre du graphique.
# best_algo correspond à l'algorithme retenu.
# k_best correspond au nombre de clusters retenu.
ax.set_title(
    f"Heatmap z-score des clusters ({best_algo}, K={k_best})"
)

# Ajustement automatique de la mise en page.
# Cela évite que les titres ou labels soient coupés.
plt.tight_layout()

# Affichage final du graphique.
plt.show()

In [26]:
# On sélectionne un échantillon de clients pour la PCA.
# len(X) correspond au nombre total de clients actifs utilisés dans le clustering.
# min(40_000, len(X)) signifie :
# - si le dataset contient plus de 40 000 clients, on prend seulement 40 000 clients ;
# - sinon, on prend tous les clients.
# replace=False signifie qu'un client ne peut pas être sélectionné plusieurs fois.
pca_idx = rng.choice(
    len(X),
    size=min(40_000, len(X)),
    replace=False
)

# Création du modèle PCA avec 2 composantes principales.
# n_components=2 signifie qu'on réduit les données à deux axes :
# - PC1 : première composante principale ;
# - PC2 : deuxième composante principale.
# random_state permet de garder des résultats reproductibles.
# fit(X[pca_idx]) entraîne la PCA sur l'échantillon sélectionné.
pca = PCA(
    n_components=2,
    random_state=RANDOM_STATE
).fit(X[pca_idx])

# Transformation des données avec la PCA.
# X_pca contient maintenant les coordonnées des clients sur les deux axes PC1 et PC2.
X_pca = pca.transform(X[pca_idx])

# Création d'une nouvelle figure pour le graphique PCA.
# figsize=(9, 6) définit la taille du graphique.
fig, ax = plt.subplots(figsize=(9, 6))

# Création d'une palette de couleurs.
# plt.cm.tab10 est une palette standard avec plusieurs couleurs distinctes.
# np.linspace(0, 1, k_best) génère autant de couleurs que de clusters.
palette = plt.cm.tab10(
    np.linspace(0, 1, k_best)
)

# Boucle sur chaque cluster existant.
# np.unique(labels) récupère la liste des clusters présents.
# sorted permet d'afficher les clusters dans l'ordre : 0, 1, 2, etc.
for cid in sorted(np.unique(labels)):

    # Création d'un masque booléen.
    # mask vaut True pour les clients appartenant au cluster cid.
    # labels[pca_idx] permet de récupérer les labels des clients utilisés dans la PCA.
    mask = labels[pca_idx] == cid

    # Affichage des clients du cluster cid sous forme de points.
    # X_pca[mask, 0] correspond aux coordonnées sur l'axe PC1.
    # X_pca[mask, 1] correspond aux coordonnées sur l'axe PC2.
    # s=4 définit une petite taille de point.
    # alpha=0.35 rend les points transparents pour mieux voir les zones denses.
    # color=palette[cid] donne une couleur différente à chaque cluster.
    # label affiche le nom du cluster dans la légende.
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        s=4,
        alpha=0.35,
        color=palette[cid],
        label=f"C{cid} — {cluster_names[cid]}"
    )

# Ajout de la légende.
# markerscale=3 agrandit les points dans la légende pour qu'ils soient visibles.
# loc="best" place automatiquement la légende au meilleur endroit.
# fontsize=8 réduit la taille du texte.
leg = ax.legend(
    markerscale=3,
    loc="best",
    fontsize=8
)

# Nom de l'axe X.
# pca.explained_variance_ratio_[0] indique le pourcentage d'information expliqué par PC1.
# :.1% affiche ce pourcentage avec un chiffre après la virgule.
ax.set_xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]:.1%})"
)

# Nom de l'axe Y.
# pca.explained_variance_ratio_[1] indique le pourcentage d'information expliqué par PC2.
ax.set_ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]:.1%})"
)

# Titre du graphique.
# Il précise que la visualisation est une projection PCA en deux dimensions.
ax.set_title(
    "Projection PCA 2D des clusters (échantillon 40k)"
)

# Ajustement automatique de la mise en page.
plt.tight_layout()

# Affichage final du graphique.
plt.show()

Les visualisations confirment la cohérence de l’interprétation des clusters.

La heatmap des z-scores montre que le cluster 1 possède un profil clairement favorable, avec un meilleur taux de remboursement, une forte proportion de clients totalement remboursés et une dette très faible. 
Ce comportement justifie son nom métier : Bon-payeur.

Le cluster 0 représente la majorité des clients actifs. 
Il est plus proche du comportement moyen global, mais présente davantage de clients avec une dette restante et moins de clients totalement remboursés. 
Il est donc nommé Standard, tout en nécessitant une surveillance sur les indicateurs liés à la dette.

La projection PCA 2D complète cette analyse en donnant une représentation visuelle des clients et de leurs clusters. 
Elle permet de vérifier la structure générale des groupes, tout en rappelant que la séparation visuelle peut être partielle à cause de la réduction des données à deux dimensions.

Ainsi, l’étape 9.1 permet de passer d’un résultat numérique de clustering à une interprétation visuelle et métier exploitable dans la plateforme décisionnelle.

## 10. Risk Ladder — Segmentation ordinale 5 niveaux

La Risk Ladder est une échelle de risque ordonnée permettant de classer chaque client selon son niveau de risque métier.

Elle contient cinq niveaux, du moins risqué au plus risqué :

| Niveau | Label | Couleur |
|---|---|---|
| 1 | Sans risque | Vert |
| 2 | Normal | Vert clair |
| 3 | À risque | Jaune |
| 4 | Haut risque | Orange |
| 5 | Blacklist | Rouge |

Cette section vient après le clustering, car elle répond à une autre question.

Le clustering réalisé dans les sections précédentes permet d’identifier le type comportemental du client.  
Il répond à la question :

**Quel type de client est-ce ?**

La Risk Ladder permet plutôt d’identifier l’urgence d’action associée à chaque client.  
Elle répond à la question :

**Quel est le niveau de risque du client ?**

Ainsi, le clustering et la Risk Ladder sont complémentaires.

Le clustering produit des informations comme `cluster_id` et `cluster_name`, tandis que la Risk Ladder produit des informations comme `risk_score_raw`, `risk_level` et `risk_label`.

La construction de cette échelle repose sur une approche hybride :

1. L’état du client (`STATE`) donne une première indication métier.
2. Un score métier composite (`risk_score_raw`) est calculé à partir de plusieurs signaux de risque.
3. Des flags comportementaux comme `never_repaid`, `has_debt` et `full_repayer` permettent d’ajuster le niveau final.
4. Les niveaux sont attribués à l’aide de règles métier explicables.

Cette approche est volontairement différente d’un clustering non supervisé pur.  
Elle permet d’obtenir une segmentation plus lisible, plus ordonnée et plus facilement exploitable par l'agent décisionnel.

Chaque niveau de risque peut être justifié par une règle claire, ce qui rend la décision plus traçable.

### 10.1 Calcul du score métier composite

Cette étape calcule un score brut de risque appelé `risk_score_raw`.

Ce score est compris entre 0 et 1 :

- une valeur proche de 0 signifie que le client présente peu de risque ;
- une valeur proche de 1 signifie que le client présente un risque élevé.

Le score est calculé à partir de six signaux métier :

| Signal | Poids de pondération | Logique métier |
|---|---:|---|
| Taux de remboursement | 28 % | Moins le client rembourse, plus il est risqué |
| Dette restante | 22 % | Plus la dette est élevée, plus le risque augmente |
| Frais restants | 12 % | Plus les frais non remboursés sont élevés, plus le risque augmente |
| Nombre de SOS | 15 % | Un usage intensif du SOS peut indiquer une dépendance au crédit d’urgence |
| Ancienneté du crédit | 13 % | Un crédit ancien non soldé peut augmenter le risque |
| Frais moyens | 10 % | Des frais élevés peuvent indiquer un usage coûteux ou répété du service |

Chaque signal est d’abord transformé en un sous-score compris entre 0 et 1.  
Cette transformation s’appelle la normalisation.

Deux fonctions de normalisation sont utilisées selon la logique métier de la variable :

- `_norm_high_is_risk` est utilisée lorsque les valeurs élevées indiquent un risque élevé ;
- `_norm_low_is_risk` est utilisée lorsque les valeurs faibles indiquent un risque élevé.

Par exemple, pour la dette restante, plus la valeur est élevée, plus le risque augmente.  
On utilise donc `_norm_high_is_risk`.

À l’inverse, pour le taux de remboursement, plus la valeur est faible, plus le risque augmente.  
On utilise donc `_norm_low_is_risk`.

Après normalisation, chaque sous-score est multiplié par un poids de pondération.  
Ces pondérations permettent de donner plus d’importance aux signaux les plus significatifs.

Le taux de remboursement a le poids le plus important, car il représente le principal indicateur de risque.  
La dette restante est également fortement pondérée, car elle mesure directement le montant encore non remboursé.

Le résultat final est stocké dans la colonne `risk_score_raw`.

Cette étape ne classe pas encore les clients dans les cinq niveaux de la Risk Ladder.  
Elle prépare seulement le score brut qui sera utilisé ensuite par les règles métier.

In [27]:
# ------------------------------------------------------------
# 10.1 Calcul du score métier composite
# ------------------------------------------------------------

# Cette fonction est utilisée quand une valeur élevée signifie un risque élevé.
# Exemple :
# - plus la dette restante est élevée, plus le risque est élevé ;
# - plus les frais restants sont élevés, plus le risque est élevé ;
# - plus le nombre de SOS est élevé, plus le risque est élevé.
def _norm_high_is_risk(x, low, high):

    # On transforme la valeur x en un score compris entre 0 et 1.
    # Si x est proche de la borne basse low, le score sera proche de 0.
    # Si x est proche de la borne haute high, le score sera proche de 1.
    #
    # Le + 1e-9 permet d’éviter une division par zéro
    # dans le cas où high et low auraient la même valeur.
    t = (x - low) / (high - low + 1e-9)

    # np.clip limite les valeurs entre 0 et 1.
    # Si t est inférieur à 0, il devient 0.
    # Si t est supérieur à 1, il devient 1.
    return np.clip(t, 0.0, 1.0)


# Cette fonction est utilisée quand une valeur faible signifie un risque élevé.
# Exemple :
# - plus le taux de remboursement est faible, plus le risque est élevé.
def _norm_low_is_risk(x, low, high):

    # Ici, la logique est inversée par rapport à _norm_high_is_risk.
    #
    # Si x est faible, le score de risque doit être élevé.
    # Si x est élevé, le score de risque doit être faible.
    t = (high - x) / (high - low + 1e-9)

    # On limite également le résultat entre 0 et 1.
    return np.clip(t, 0.0, 1.0)


# Fonction principale qui calcule le score métier de risque.
# Elle prend en entrée un DataFrame pandas.
# Elle retourne une Series pandas contenant un score de risque pour chaque client.
def compute_rule_score(frame: pd.DataFrame) -> pd.Series:

    # ------------------------------------------------------------
    # 1) Récupération des variables métier
    # ------------------------------------------------------------

    # Taux moyen de remboursement.
    # Plus ce taux est faible, plus le client est risqué.
    reimburse_ratio = frame["AVG_REIMBURSE_RATIO"].to_numpy(dtype=float)

    # Montant total restant à rembourser.
    # Plus ce montant est élevé, plus le risque augmente.
    outstanding = frame["TOTAL_OUTSTANDING_AMOUNT"].to_numpy(dtype=float)

    # Frais restants non remboursés.
    # Plus ces frais sont élevés, plus le risque augmente.
    outstanding_fee = frame["TOTAL_OUTSTANDING_FEE"].to_numpy(dtype=float)

    # Nombre d’utilisations du service SOS.
    # Un nombre élevé de SOS peut indiquer une dépendance au crédit d’urgence.
    nb_sos = frame["NB_SOS"].to_numpy(dtype=float)

    # Délai moyen depuis le crédit.
    # Un crédit ancien non soldé peut indiquer un risque plus important.
    days_since = frame["AVG_DAYS_SINCE_CREDIT"].to_numpy(dtype=float)

    # Frais moyens de crédit.
    # Des frais élevés peuvent indiquer un usage coûteux ou répété du service.
    avg_fee = frame["AVG_CREDIT_FEE"].to_numpy(dtype=float)

    # ------------------------------------------------------------
    # 2) Normalisation des variables en sous-scores entre 0 et 1
    # ------------------------------------------------------------

    # Sous-score 1 : taux de remboursement.
    #
    # Ici, une valeur faible est risquée.
    # Donc on utilise _norm_low_is_risk.
    #
    # Exemple :
    # - remboursement faible  -> score proche de 1 ;
    # - remboursement élevé   -> score proche de 0.
    s_ratio = _norm_low_is_risk(
        reimburse_ratio,
        low=0.15,
        high=0.85
    )

    # Sous-score 2 : dette restante.
    #
    # Ici, une valeur élevée est risquée.
    # Donc on utilise _norm_high_is_risk.
    #
    # Les bornes sont calculées avec les percentiles 10 et 90.
    # Cela permet de limiter l’impact des valeurs extrêmes.
    s_out = _norm_high_is_risk(
        outstanding,
        low=np.percentile(outstanding, 10),
        high=np.percentile(outstanding, 90)
    )

    # Sous-score 3 : frais restants.
    #
    # Plus les frais non remboursés sont élevés,
    # plus le risque augmente.
    s_fee_out = _norm_high_is_risk(
        outstanding_fee,
        low=np.percentile(outstanding_fee, 10),
        high=np.percentile(outstanding_fee, 90)
    )

    # Sous-score 4 : nombre d’utilisations SOS.
    #
    # Plus le client utilise SOS, plus le risque augmente.
    #
    # La borne basse est 0.
    # La borne haute est le maximum entre :
    # - 5 ;
    # - le percentile 95 du nombre de SOS.
    #
    # Cela permet d’avoir une limite métier minimale,
    # tout en s’adaptant à la distribution réelle des données.
    s_sos = _norm_high_is_risk(
        nb_sos,
        low=0.0,
        high=max(5.0, float(np.percentile(nb_sos, 95)))
    )

    # Sous-score 5 : ancienneté du crédit.
    #
    # Plus le crédit est ancien, plus le risque peut augmenter.
    #
    # La borne haute est le maximum entre :
    # - 60 jours ;
    # - le percentile 90 de l’ancienneté du crédit.
    s_days = _norm_high_is_risk(
        days_since,
        low=0.0,
        high=max(60.0, float(np.percentile(days_since, 90)))
    )

    # Sous-score 6 : frais moyens de crédit.
    #
    # Plus les frais moyens sont élevés,
    # plus le client peut être considéré comme risqué.
    s_fee = _norm_high_is_risk(
        avg_fee,
        low=np.percentile(avg_fee, 10),
        high=np.percentile(avg_fee, 90)
    )

    # ------------------------------------------------------------
    # 3) Définition des poids de pondération
    # ------------------------------------------------------------

    # Les poids de pondération indiquent l’importance de chaque signal
    # dans le calcul du score final.
    #
    # La somme des poids est égale à 1.
    #
    # L’ordre des poids doit correspondre exactement à l’ordre des sous-scores :
    # s_ratio, s_out, s_fee_out, s_sos, s_days, s_fee.
    weights = np.array([
        0.28,  # taux de remboursement
        0.22,  # dette restante
        0.12,  # frais restants
        0.15,  # nombre de SOS
        0.13,  # ancienneté du crédit
        0.10   # frais moyens
    ])

    # ------------------------------------------------------------
    # 4) Regroupement des sous-scores
    # ------------------------------------------------------------

    # On regroupe les six sous-scores dans une matrice.
    #
    # Chaque ligne représente un signal.
    # Chaque colonne représente un client.
    parts = np.vstack([
        s_ratio,
        s_out,
        s_fee_out,
        s_sos,
        s_days,
        s_fee
    ])

    # ------------------------------------------------------------
    # 5) Calcul du score final
    # ------------------------------------------------------------

    # parts.T transpose la matrice pour avoir :
    # - une ligne par client ;
    # - une colonne par signal.
    #
    # Chaque sous-score est multiplié par son poids de pondération.
    # Ensuite, on additionne les six contributions pour obtenir
    # le score brut de risque de chaque client.
    score = (parts.T * weights).sum(axis=1)

    # On s’assure que le score final reste compris entre 0 et 1.
    # Puis on retourne le résultat sous forme de Series pandas,
    # en conservant le même index que le DataFrame initial.
    return pd.Series(
        np.clip(score, 0.0, 1.0),
        index=frame.index
    )


# ------------------------------------------------------------
# 6) Application du score au DataFrame principal
# ------------------------------------------------------------

# On applique la fonction compute_rule_score au DataFrame df.
# Une nouvelle colonne est créée : risk_score_raw.
#
# Cette colonne contient le score brut de risque pour chaque client.
df["risk_score_raw"] = compute_rule_score(df)


# ------------------------------------------------------------
# 7) Affichage des statistiques du score
# ------------------------------------------------------------

# On affiche un titre pour indiquer que l’on va analyser
# la distribution du score brut de risque.
print("risk_score_raw — distribution :")

# describe() affiche les principales statistiques :
# - count : nombre de clients ;
# - mean : moyenne ;
# - std : écart-type ;
# - min : valeur minimale ;
# - 25% : premier quartile ;
# - 50% : médiane ;
# - 75% : troisième quartile ;
# - max : valeur maximale.
#
# round(3) arrondit les résultats à trois décimales.
print(df["risk_score_raw"].describe().round(3))


# ------------------------------------------------------------
# 8) Visualisation de la distribution du score
# ------------------------------------------------------------

# Création d’une figure matplotlib.
# figsize=(8, 3) définit la taille du graphique.
fig, ax = plt.subplots(figsize=(8, 3))

# Création de l’histogramme.
#
# L’histogramme montre combien de clients se trouvent
# dans chaque intervalle de score.
#
# bins=60 divise l’axe des scores en 60 intervalles.
ax.hist(
    df["risk_score_raw"],
    bins=60,
    color="steelblue",
    edgecolor="white"
)

# Nom de l’axe horizontal.
# Il représente le score brut de risque.
ax.set_xlabel("risk_score_raw")

# Nom de l’axe vertical.
# Il représente le nombre de clients.
ax.set_ylabel("Clients")

# Titre du graphique.
ax.set_title("Distribution du score métier composite sur tout le parc")

# Ajustement automatique de la mise en page.
plt.tight_layout()

# Affichage du graphique.
plt.show()

risk_score_raw — distribution :
count   9748.000
mean       0.293
std        0.177
min        0.013
25%        0.154
50%        0.239
75%        0.409
max        0.862
Name: risk_score_raw, dtype: float64


L’histogramme représente la distribution du score `risk_score_raw` sur l’ensemble du parc client.

L’axe horizontal correspond au score brut de risque, compris entre 0 et 1.  
L’axe vertical correspond au nombre de clients dans chaque intervalle de score.

Un score proche de 0 indique un client avec peu de signaux de risque.  
Un score proche de 1 indique un client présentant plusieurs signaux défavorables.

Cette étape permet donc d’obtenir une première mesure quantitative du risque client.  
Ce score sera ensuite utilisé avec les règles métier pour attribuer les niveaux finaux de la Risk Ladder.

### 10.2 Score final ordinal par bandes STATE

Cette étape calcule un score final de risque appelé `risk_score_final`.

Dans l’étape précédente, nous avons calculé un score brut `risk_score_raw`, basé sur les signaux métier du client : remboursement, dette, frais, usage SOS, ancienneté du crédit et frais moyens.

Cependant, ce score brut ne suffit pas à lui seul, car l’état du client (`STATE`) porte une information métier très importante.

Par exemple, un client `DISCONNECTED` doit toujours être considéré plus risqué qu’un client `ACTIVE`, même si leurs scores bruts sont proches.

Pour cette raison, chaque état client est associé à une bande de score réservée :

| STATE | Bande de score |
|---|---|
| ACTIVE | 0.00 à 0.70 |
| ON-HOLD | 0.70 à 0.80 |
| SUSPENDED | 0.85 à 0.93 |
| DISCONNECTED | 0.93 à 1.00 |
| OTHER | 0.70 à 0.80 |

Le score brut `risk_score_raw` est ensuite projeté à l’intérieur de la bande correspondant au `STATE` du client.

La formule utilisée est :

`risk_score_final = lo + (hi - lo) × risk_score_raw`

où :

- `lo` représente le début de la bande associée au STATE ;
- `hi` représente la fin de la bande associée au STATE ;
- `risk_score_raw` permet d’ordonner les clients à l’intérieur de cette bande.

Cette approche garantit une hiérarchie métier claire entre les états clients.

Ainsi, les clients `DISCONNECTED` restent toujours au-dessus des clients `ACTIVE`, tandis que le score brut permet de classer finement les clients à l’intérieur de chaque état.

In [28]:
# ------------------------------------------------------------
# 10.2 Score final ordinal par bandes STATE
# ------------------------------------------------------------

# On définit les bandes de score réservées pour chaque état client.
#
# Chaque STATE possède une plage de score fixe.
# Cela permet de respecter une hiérarchie métier :
#
# ACTIVE       -> risque le plus faible
# ON-HOLD      -> risque plus élevé
# SUSPENDED    -> risque important
# DISCONNECTED -> risque maximal
#
# Exemple :
# Un client ACTIVE aura toujours un score final entre 0.00 et 0.70.
# Un client DISCONNECTED aura toujours un score final entre 0.93 et 1.00.
STATE_BANDS = {
    "ACTIVE":       (0.00, 0.70),
    "ON-HOLD":      (0.70, 0.80),
    "SUSPENDED":    (0.85, 0.93),
    "DISCONNECTED": (0.93, 1.00),
    "OTHER":        (0.70, 0.80),
}


# ------------------------------------------------------------
# 1) Récupération de la borne basse de chaque STATE
# ------------------------------------------------------------

# Pour chaque client, on regarde son STATE.
# Ensuite, on récupère le début de la bande associée à ce STATE.
#
# Exemple :
# - si STATE = ACTIVE, alors lo = 0.00 ;
# - si STATE = ON-HOLD, alors lo = 0.70 ;
# - si STATE = DISCONNECTED, alors lo = 0.93.
#
# STATE_BANDS.get(s, STATE_BANDS["OTHER"]) signifie :
# - si le STATE existe dans le dictionnaire, on utilise sa bande ;
# - sinon, on utilise la bande OTHER par défaut.
#
# [0] permet de récupérer la première valeur de la bande, c’est-à-dire la borne basse.
lo = df["STATE"].map(
    lambda s: STATE_BANDS.get(s, STATE_BANDS["OTHER"])[0]
).to_numpy()


# ------------------------------------------------------------
# 2) Récupération de la borne haute de chaque STATE
# ------------------------------------------------------------

# Même logique que pour lo.
# Ici, on récupère la fin de la bande associée au STATE.
#
# Exemple :
# - si STATE = ACTIVE, alors hi = 0.70 ;
# - si STATE = ON-HOLD, alors hi = 0.80 ;
# - si STATE = DISCONNECTED, alors hi = 1.00.
#
# [1] permet de récupérer la deuxième valeur de la bande, c’est-à-dire la borne haute.
hi = df["STATE"].map(
    lambda s: STATE_BANDS.get(s, STATE_BANDS["OTHER"])[1]
).to_numpy()


# ------------------------------------------------------------
# 3) Calcul du score final
# ------------------------------------------------------------

# On calcule risk_score_final avec la formule :
#
# risk_score_final = lo + (hi - lo) * risk_score_raw
#
# Explication :
# - lo donne le début de la bande ;
# - hi - lo donne la largeur de la bande ;
# - risk_score_raw place le client à l’intérieur de cette bande.
#
# Exemple :
# Si le client est ACTIVE :
# lo = 0.00
# hi = 0.70
# risk_score_raw = 0.50
#
# risk_score_final = 0.00 + (0.70 - 0.00) * 0.50
# risk_score_final = 0.35
#
# Donc le client reste bien dans la bande ACTIVE.
df["risk_score_final"] = lo + (hi - lo) * df["risk_score_raw"].to_numpy()


# ------------------------------------------------------------
# 4) Vérification des scores par STATE
# ------------------------------------------------------------

# On affiche un message pour annoncer la vérification.
print("risk_score_final — range par STATE (min / max) :")


# On regroupe les clients par STATE.
# Puis on calcule les statistiques du score final pour chaque STATE :
#
# min   -> score final minimum dans ce STATE ;
# max   -> score final maximum dans ce STATE ;
# mean  -> score final moyen dans ce STATE ;
# count -> nombre de clients dans ce STATE.
#
# round(3) arrondit les résultats à 3 décimales.
print(
    df.groupby("STATE")["risk_score_final"]
      .agg(["min", "max", "mean", "count"])
      .round(3)
)

risk_score_final — range par STATE (min / max) :
               min   max  mean  count
STATE                                
ACTIVE       0.015 0.603 0.205   9208
DISCONNECTED 0.940 0.988 0.958     28
ON-HOLD      0.729 0.751 0.743     15
SUSPENDED    0.851 0.919 0.872    497


### 10.3 Assignation du niveau par règles métier

- Cette étape attribue à chaque client un niveau final de risque selon la Risk Ladder.

- Les niveaux possibles sont :

| Niveau | Label |
|---|---|
| 1 | Sans risque |
| 2 | Normal |
| 3 | À risque |
| 4 | Haut risque |
| 5 | Blacklist |

- L’assignation ne repose pas sur des pourcentages imposés.
- Les proportions finales dépendent directement des données et des règles métier.

- Les informations utilisées sont :
  - `STATE` : état administratif du client ;
  - `risk_score_raw` : score brut de risque ;
  - `never_repaid` : client qui n’a jamais remboursé ;
  - `has_debt` : client avec dette restante ;
  - `full_repayer` : client qui rembourse totalement.

- Les règles sont appliquées dans l’ordre, du cas le plus grave au moins grave.

- Règles principales :
  - `SUSPENDED`, `DISCONNECTED` ou `OTHER` → **Blacklist**
  - `ON-HOLD` → **Haut risque**
  - `ACTIVE` avec `never_repaid = 1` ou `risk_score_raw >= 0.65` → **Haut risque**
  - `ACTIVE` avec `has_debt = 1` et `risk_score_raw >= 0.35` → **À risque**
  - `ACTIVE` avec `full_repayer = 1`, `has_debt = 0` et `risk_score_raw < 0.20` → **Sans risque**
  - Tous les autres cas → **Normal**

- Cette logique permet d’obtenir une classification :
  - claire ;
  - ordonnée ;
  - explicable ;
  - exploitable par l'agent.

In [29]:
# ------------------------------------------------------------
# 10.3 Assignation du niveau par règles métier
# ------------------------------------------------------------

# Dictionnaire qui associe chaque niveau numérique à son label métier.
#
# Exemple :
# 1 correspond à "Sans risque"
# 5 correspond à "Blacklist"
LABELS = {
    1: "Sans risque",
    2: "Normal",
    3: "A risque",
    4: "Haut risque",
    5: "Blacklist",
}


# ------------------------------------------------------------
# Définition des seuils métier
# ------------------------------------------------------------

# Seuil haut du score brut.
# Si un client actif a un score supérieur ou égal à 0.65,
# il est considéré comme fortement risqué.
HIGH_SCORE_THR = 0.65

# Seuil moyen du score brut.
# Si un client actif a une dette et un score supérieur ou égal à 0.35,
# il est considéré comme "À risque".
MID_SCORE_THR = 0.35

# Seuil bas du score brut.
# Si un client actif rembourse totalement, n’a pas de dette,
# et possède un score inférieur à 0.20,
# il est considéré comme "Sans risque".
LOW_SCORE_THR = 0.20


# ------------------------------------------------------------
# Fonction d’assignation du niveau de risque
# ------------------------------------------------------------

# Cette fonction reçoit une ligne du DataFrame.
# Une ligne correspond à un client.
#
# Elle retourne un niveau de risque entre 1 et 5.
def assign_level_rules(row):

    # On récupère l’état administratif du client.
    # Exemple : ACTIVE, ON-HOLD, SUSPENDED, DISCONNECTED.
    state = row["STATE"]

    # On récupère le score brut de risque calculé en 10.1.
    score = row["risk_score_raw"]


    # ------------------------------------------------------------
    # Règle 1 : Blacklist
    # ------------------------------------------------------------

    # Si le client est suspendu, déconnecté ou dans un état autre/non reconnu,
    # il est directement classé au niveau 5.
    #
    # C’est le niveau le plus grave.
    if state in ("SUSPENDED", "DISCONNECTED", "OTHER"):
        return 5


    # ------------------------------------------------------------
    # Règle 2 : Haut risque lié au STATE
    # ------------------------------------------------------------

    # Si le client est ON-HOLD,
    # il est classé directement en Haut risque.
    #
    # ON-HOLD signifie que le compte est en attente ou bloqué temporairement.
    if state == "ON-HOLD":
        return 4


    # ------------------------------------------------------------
    # Règle 3 : Haut risque lié au comportement
    # ------------------------------------------------------------

    # Si le client est actif,
    # mais qu’il n’a jamais remboursé,
    # ou que son score brut est très élevé,
    # alors il est classé en Haut risque.
    #
    # row.get("never_repaid", 0) permet de récupérer la valeur never_repaid.
    # Si la colonne ou la valeur n’existe pas, on utilise 0 par défaut.
    if state == "ACTIVE" and (
        row.get("never_repaid", 0) == 1
        or score >= HIGH_SCORE_THR
    ):
        return 4


    # ------------------------------------------------------------
    # Règle 4 : À risque
    # ------------------------------------------------------------

    # Si le client est actif,
    # qu’il possède une dette,
    # et que son score brut est au moins moyen,
    # alors il est classé "À risque".
    #
    # has_debt = 1 signifie que le client a une dette restante.
    if (
        state == "ACTIVE"
        and row.get("has_debt", 0) == 1
        and score >= MID_SCORE_THR
    ):
        return 3


    # ------------------------------------------------------------
    # Règle 5 : Sans risque
    # ------------------------------------------------------------

    # Si le client est actif,
    # qu’il rembourse totalement,
    # qu’il n’a aucune dette,
    # et que son score brut est faible,
    # alors il est classé "Sans risque".
    #
    # full_repayer = 1 signifie que le client rembourse totalement.
    # has_debt = 0 signifie qu’il n’a pas de dette.
    if (
        state == "ACTIVE"
        and row.get("full_repayer", 0) == 1
        and row.get("has_debt", 0) == 0
        and score < LOW_SCORE_THR
    ):
        return 1


    # ------------------------------------------------------------
    # Règle 6 : Normal par défaut
    # ------------------------------------------------------------

    # Si le client ne déclenche aucune des règles précédentes,
    # il est classé en niveau 2 : Normal.
    #
    # Cela concerne surtout les clients actifs qui n’ont pas de signal fort.
    return 2


# ------------------------------------------------------------
# Application des règles à tous les clients
# ------------------------------------------------------------

# On applique la fonction assign_level_rules à chaque ligne du DataFrame.
#
# axis=1 signifie que la fonction est appliquée ligne par ligne.
# Chaque ligne correspond à un client.
#
# Le résultat est stocké dans une nouvelle colonne risk_level.
df["risk_level"] = df.apply(assign_level_rules, axis=1).astype(int)


# On transforme le niveau numérique en label métier.
#
# Exemple :
# 1 devient "Sans risque"
# 4 devient "Haut risque"
# 5 devient "Blacklist"
df["risk_label"] = df["risk_level"].map(LABELS)


# ------------------------------------------------------------
# Calcul de la distribution réelle des niveaux
# ------------------------------------------------------------

# On compte combien de clients appartiennent à chaque niveau.
#
# value_counts() compte les occurrences de chaque niveau.
# sort_index() trie les niveaux dans l’ordre : 1, 2, 3, 4, 5.
obtained = df["risk_level"].value_counts().sort_index()


# On calcule le pourcentage de clients dans chaque niveau.
#
# obtained / len(df) donne la proportion.
# * 100 transforme la proportion en pourcentage.
# round(2) arrondit à deux décimales.
obtained_pct = (obtained / len(df) * 100).round(2)


# ------------------------------------------------------------
# Création d’un tableau récapitulatif
# ------------------------------------------------------------

# On construit un DataFrame appelé distrib.
#
# Il contient :
# - le niveau ;
# - le label ;
# - le nombre de clients ;
# - le pourcentage de clients.
distrib = pd.DataFrame({
    "level": [k for k in range(1, 6)],
    "label": [LABELS[k] for k in range(1, 6)],
    "count": [int(obtained.get(k, 0)) for k in range(1, 6)],
    "pct": [obtained_pct.get(k, 0.0) for k in range(1, 6)],
})


# ------------------------------------------------------------
# Affichage du résultat
# ------------------------------------------------------------

# On affiche un message explicatif.
print("Distribution réelle du Risk Ladder (selon règles métier) :")

# Affichage du tableau final.
distrib

Distribution réelle du Risk Ladder (selon règles métier) :


### 10.4 Visualisation du Risk Ladder

- Cette étape visualise la répartition des clients selon les 5 niveaux de risque.

- Deux graphiques sont utilisés :
  - un camembert pour voir la proportion de chaque niveau ;
  - un graphique en barres horizontales pour voir le volume exact de clients.

- Les couleurs suivent une logique métier :
  - vert : risque faible ;
  - vert clair : normal ;
  - jaune : à risque ;
  - orange : haut risque ;
  - rouge : blacklist.

- Cette visualisation permet de vérifier rapidement :
  - combien de clients sont sans risque ;
  - combien sont normaux ;
  - combien nécessitent une surveillance ;
  - combien sont en haut risque ou blacklist.

- Les proportions affichées ne sont pas imposées.
- Elles viennent directement des règles métier appliquées en section 10.3.

In [30]:
# ------------------------------------------------------------
# 10.4 Visualisation du Risk Ladder
# ------------------------------------------------------------

# Dictionnaire des couleurs associées à chaque niveau de risque.
#
# La logique va du vert vers le rouge :
# - vert       : risque faible ;
# - jaune      : risque intermédiaire ;
# - orange     : haut risque ;
# - rouge      : blacklist.
COLORS = {
    1: "#3DBB96",  # Sans risque
    2: "#88D26B",  # Normal
    3: "#F4D849",  # À risque
    4: "#F39C3A",  # Haut risque
    5: "#C0392B",  # Blacklist
}


# ------------------------------------------------------------
# 1) Calcul du nombre de clients par niveau
# ------------------------------------------------------------

# On compte le nombre de clients dans chaque niveau de risque.
#
# value_counts() compte les occurrences de chaque risk_level.
# sort_index() trie les niveaux dans l’ordre : 1, 2, 3, 4, 5.
level_counts = df["risk_level"].value_counts().sort_index()


# ------------------------------------------------------------
# 2) Calcul du pourcentage par niveau
# ------------------------------------------------------------

# On calcule la proportion de chaque niveau.
#
# level_counts / len(df) donne la proportion.
# * 100 transforme la proportion en pourcentage.
# round(2) arrondit à deux chiffres après la virgule.
level_pct = (level_counts / len(df) * 100).round(2)


# ------------------------------------------------------------
# 3) Préparation des couleurs et de la légende
# ------------------------------------------------------------

# On récupère la couleur de chaque niveau présent dans les données.
#
# Exemple :
# si les niveaux présents sont 1, 2, 3, 4, 5,
# palette contiendra les 5 couleurs correspondantes.
palette = [
    COLORS[k]
    for k in level_counts.index
]


# Création des textes de légende.
#
# Exemple :
# "1 - Sans risque : 23.98%"
legend_labels = [
    f"{k} - {LABELS[k]} : {level_pct[k]:.2f}%"
    for k in level_counts.index
]


# ------------------------------------------------------------
# 4) Création de la figure
# ------------------------------------------------------------

# On crée une figure avec 2 graphiques côte à côte.
#
# 1 ligne, 2 colonnes :
# - axes[0] : camembert ;
# - axes[1] : barres horizontales.
#
# figsize=(13, 5) définit la taille globale de la figure.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))


# ------------------------------------------------------------
# 5) Camembert : répartition en pourcentage
# ------------------------------------------------------------

# Le camembert permet de visualiser rapidement la proportion
# de chaque niveau de risque.
axes[0].pie(
    level_counts.values,              # valeurs à afficher
    colors=palette,                   # couleurs des niveaux
    startangle=90,                    # rotation du camembert
    wedgeprops=dict(
        edgecolor="white",            # bordure blanche entre les parts
        linewidth=2                   # épaisseur de la bordure
    ),
    autopct=lambda p: f"{p:.1f}%",    # affichage du pourcentage
    textprops=dict(
        color="white",                # couleur du texte
        fontsize=10,                  # taille du texte
        fontweight="bold"             # texte en gras
    )
)

# Titre du camembert.
axes[0].set_title("Répartition Risk Ladder")


# Ajout de la légende à côté du camembert.
#
# loc="center left" place la légende à gauche du point d’ancrage.
# bbox_to_anchor=(1.0, 0.5) place la légende à droite du graphique.
# frameon=False enlève le cadre autour de la légende.
axes[0].legend(
    legend_labels,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    fontsize=9,
    frameon=False
)


# ------------------------------------------------------------
# 6) Graphique en barres horizontales
# ------------------------------------------------------------

# On crée un graphique en barres horizontales.
#
# range(len(level_counts))[::-1] permet d’afficher les niveaux
# du haut vers le bas dans le même ordre visuel que la Risk Ladder.
bars = axes[1].barh(
    range(len(level_counts))[::-1],
    level_counts.values,
    color=palette,
    edgecolor="white"
)


# On définit les positions des labels sur l’axe Y.
axes[1].set_yticks(
    range(len(level_counts))[::-1]
)


# On affiche les noms des niveaux sur l’axe Y.
#
# Exemple :
# Sans risque, Normal, À risque, Haut risque, Blacklist.
axes[1].set_yticklabels(
    [LABELS[k] for k in level_counts.index],
    fontsize=10
)


# ------------------------------------------------------------
# 7) Ajout des valeurs sur les barres
# ------------------------------------------------------------

# Pour chaque barre, on affiche :
# - le nombre de clients ;
# - le pourcentage correspondant.
for i, (k, v) in enumerate(level_counts.items()):

    # v représente le nombre de clients du niveau k.
    # level_pct[k] représente le pourcentage de ce niveau.
    #
    # len(level_counts) - 1 - i permet de placer le texte
    # sur la bonne barre.
    axes[1].text(
        v,
        len(level_counts) - 1 - i,
        f"  {v:,} ({level_pct[k]:.2f}%)",
        va="center",
        fontsize=9
    )


# Nom de l’axe horizontal.
axes[1].set_xlabel("Nombre de clients")


# Titre du graphique en barres.
axes[1].set_title("Volume par niveau de risque")


# ------------------------------------------------------------
# 8) Affichage final
# ------------------------------------------------------------

# Ajustement automatique de la mise en page.
plt.tight_layout()

# Affichage des deux graphiques.
plt.show()

### 10.5 Cohérence Risk Level × STATE × Cluster ML

- Cette étape sert à vérifier que les niveaux de risque attribués sont cohérents.

- Deux contrôles sont réalisés :

1. **STATE × risk_label**
   - On vérifie la répartition des niveaux de risque selon l’état administratif du client.
   - Exemple attendu :
     - `DISCONNECTED` doit être principalement en `Blacklist`.
     - `SUSPENDED` doit être en `Blacklist`.
     - `ON-HOLD` doit être en `Haut risque`.
     - `ACTIVE` peut être réparti entre `Sans risque`, `Normal`, `À risque` et parfois `Haut risque`.

2. **cluster_name × risk_label sur les clients ACTIVE**
   - On compare les clusters issus du ML avec les niveaux de risque.
   - Les clusters de type `Bon-payeur` doivent surtout apparaître dans les niveaux `Sans risque` ou `Normal`.
   - Les clusters plus standards ou moins favorables peuvent se répartir entre `Normal`, `À risque` et `Haut risque`.

- L’objectif n’est pas de recalculer les niveaux.
- L’objectif est seulement de vérifier que la classification finale est logique.

- Cette étape permet donc de valider la cohérence entre :
  - l’état administratif du client ;
  - le niveau de risque final ;
  - le profil comportemental obtenu par clustering.

In [31]:
# ------------------------------------------------------------
# 10.5 Cohérence Risk Level × STATE × Cluster ML
# ------------------------------------------------------------

# ------------------------------------------------------------
# 1) Tableau croisé STATE × risk_label
# ------------------------------------------------------------

# On crée un tableau croisé entre :
# - les lignes : STATE du client ;
# - les colonnes : risk_label, c’est-à-dire le niveau de risque final.
#
# pd.crosstab permet de compter ou comparer deux variables catégorielles.
#
# normalize="index" signifie que les proportions sont calculées ligne par ligne.
# Donc chaque ligne fait 100 %.
#
# Exemple :
# pour STATE = ACTIVE, on regarde quelle proportion est :
# - Sans risque ;
# - Normal ;
# - À risque ;
# - Haut risque ;
# - Blacklist.
crosstab_state = pd.crosstab(
    df["STATE"],
    df["risk_label"],
    normalize="index"
)


# On réordonne les colonnes selon l’ordre logique de la Risk Ladder :
# 1 -> Sans risque
# 2 -> Normal
# 3 -> À risque
# 4 -> Haut risque
# 5 -> Blacklist
#
# reindex permet de forcer cet ordre.
# fillna(0) remplace les valeurs manquantes par 0.
# Cela évite d’avoir des cellules vides si un niveau n’existe pas pour un STATE.
crosstab_state = crosstab_state.reindex(
    columns=[LABELS[k] for k in range(1, 6)]
).fillna(0)


# On affiche un message explicatif.
print("STATE × risk_label (proportions par ligne, % dans chaque STATE) :")


# On multiplie par 100 pour afficher les proportions en pourcentage.
# round(1) arrondit à un chiffre après la virgule.
(crosstab_state * 100).round(1)

STATE × risk_label (proportions par ligne, % dans chaque STATE) :


In [32]:
# ------------------------------------------------------------
# 2) Préparation des clients ACTIVE pour le contrôle ML
# ------------------------------------------------------------

# On crée un masque pour sélectionner uniquement les clients ACTIVE.
active_mask = df["STATE"] == "ACTIVE"


# On récupère les colonnes de risque calculées dans df.
# Ces colonnes ont été créées dans les étapes 10.1, 10.2 et 10.3.
risk_cols = [
    "risk_score_raw",
    "risk_score_final",
    "risk_level",
    "risk_label"
]


# On copie df_active pour éviter les problèmes de modification directe.
# df_active contient normalement les clients ACTIVE utilisés dans le clustering ML.
df_active = df_active.copy()


# On ajoute dans df_active les colonnes de risque calculées dans df.
#
# On utilise df_active.index pour garder le bon alignement entre les clients.
# Cela permet de copier les bonnes valeurs de risque au bon client.
df_active[risk_cols] = df.loc[df_active.index, risk_cols]


# ------------------------------------------------------------
# 3) Tableau croisé cluster_name × risk_label
# ------------------------------------------------------------

# On crée un tableau croisé entre :
# - les lignes : cluster_name, c’est-à-dire le nom du cluster ML ;
# - les colonnes : risk_label, c’est-à-dire le niveau de risque.
#
# normalize="index" signifie que chaque ligne est transformée en pourcentage.
# Donc pour chaque cluster, on regarde comment ses clients sont répartis
# entre les niveaux de risque.
crosstab_ml = pd.crosstab(
    df_active["cluster_name"],
    df_active["risk_label"],
    normalize="index"
)


# On réordonne les colonnes dans l’ordre logique de la Risk Ladder.
# fillna(0) remplace les niveaux absents par 0.
crosstab_ml = crosstab_ml.reindex(
    columns=[LABELS[k] for k in range(1, 6)]
).fillna(0)


# On affiche un message explicatif.
print("cluster_name (ML) × risk_label sur ACTIVE (proportions par ligne) :")


# On affiche les proportions en pourcentage.
# round(1) arrondit à un chiffre après la virgule.
(crosstab_ml * 100).round(1)

cluster_name (ML) × risk_label sur ACTIVE (proportions par ligne) :


## 11. Anomalies + dataset final

- Cette section détecte les comportements atypiques parmi les clients actifs.

- La détection d’anomalies est faite avec l’algorithme `IsolationForest`.

- L’algorithme utilise le même jeu de variables transformées `X` que le clustering.

- Cela permet de garder une cohérence :
  - le clustering segmente les clients dans cet espace ;
  - l’Isolation Forest détecte les clients anormaux dans ce même espace.

- Le paramètre `contamination=0.03` signifie que l’on demande au modèle d’identifier environ 3 % des clients actifs comme anomalies.(“je pense qu’environ 3 % des clients sont atypiques”.)



- Le résultat est ajouté dans `df_active` avec deux colonnes :
  - `anomaly_flag` : indique si le client est anomalie ou non ;
  - `anomaly_score` : mesure le degré d’anomalie.

- `anomaly_flag = 1` signifie que le client est détecté comme anomalie.
- `anomaly_flag = 0` signifie que le client est considéré comme normal.

- Cette détection ne remplace pas le cluster ou le niveau de risque.
- Elle ajoute une information complémentaire.

- À la fin, on analyse aussi les anomalies par cluster pour voir si certains profils contiennent plus de comportements atypiques.

In [33]:
# ------------------------------------------------------------
# 11. Anomalies + dataset final
# ------------------------------------------------------------

# On crée un modèle Isolation Forest.
#
# Isolation Forest est un algorithme utilisé pour détecter les anomalies.
# Il cherche les points qui sont très différents des autres clients.
iso = IsolationForest(

    # contamination=0.03 signifie que l’on suppose qu’environ 3 %
    # des clients actifs sont atypiques.
    #
    # Le modèle va donc essayer d’isoler environ 3 % d’anomalies.
    contamination=0.03,

    # random_state permet de rendre les résultats reproductibles.
    # Avec la même valeur, on obtient les mêmes résultats à chaque exécution.
    random_state=RANDOM_STATE,

    # n_estimators correspond au nombre d’arbres utilisés par Isolation Forest.
    # Plus il y a d’arbres, plus le modèle est stable.
    n_estimators=200,

    # max_samples définit le nombre maximal de clients utilisés
    # pour construire chaque arbre.
    #
    # On prend au maximum 5000 clients.
    # Si le dataset contient moins de 5000 clients, on prend tous les clients.
    max_samples=min(5000, len(X)),

    # n_jobs=1 signifie que le calcul est fait sur un seul cœur CPU.
    # Cela évite certains problèmes de parallélisation selon l’environnement.
    n_jobs=1,
)


# ------------------------------------------------------------
# 1) Entraînement du modèle d’anomalies
# ------------------------------------------------------------

# On entraîne Isolation Forest sur X.
#
# X est le même espace transformé que celui utilisé pour le clustering.
# Cela garantit que les anomalies sont détectées dans le même espace
# que celui utilisé pour segmenter les clients.
iso.fit(X)


# ------------------------------------------------------------
# 2) Détection des anomalies
# ------------------------------------------------------------

# iso.predict(X) retourne :
# - 1  pour les clients considérés comme normaux ;
# - -1 pour les clients considérés comme anomalies.
#
# On transforme ensuite le résultat en 0 / 1 :
# - True devient 1 ;
# - False devient 0.
#
# Donc :
# anomaly_flag = 1 signifie anomalie ;
# anomaly_flag = 0 signifie client normal.
df_active["anomaly_flag"] = (
    iso.predict(X) == -1
).astype(int)


# ------------------------------------------------------------
# 3) Calcul du score d’anomalie
# ------------------------------------------------------------

# iso.score_samples(X) donne un score d’anomalie.
#
# Dans Isolation Forest, les scores les plus faibles correspondent
# généralement aux observations les plus atypiques.
#
# On met un signe moins devant :
# -iso.score_samples(X)
#
# Cela permet d’avoir une lecture plus intuitive :
# plus anomaly_score est élevé, plus le client est atypique.
df_active["anomaly_score"] = -iso.score_samples(X)


# ------------------------------------------------------------
# 4) Affichage du nombre total d’anomalies détectées
# ------------------------------------------------------------

# df_active["anomaly_flag"].sum() compte le nombre de clients
# détectés comme anomalies.
#
# df_active["anomaly_flag"].mean() donne la proportion d’anomalies.
#
# Comme anomaly_flag vaut 1 pour une anomalie et 0 sinon,
# la moyenne correspond directement au pourcentage d’anomalies.
print(
    f"Anomalies détectées : {df_active['anomaly_flag'].sum():,} "
    f"({df_active['anomaly_flag'].mean():.2%})"
)


# ------------------------------------------------------------
# 5) Analyse des anomalies par cluster
# ------------------------------------------------------------

# On affiche un titre.
print("\nAnomalies par cluster :")


# On regroupe les clients actifs par cluster.
#
# Les colonnes utilisées sont :
# - cluster_id : identifiant numérique du cluster ;
# - cluster_name : nom métier du cluster.
#
# Ensuite, on calcule :
# - total : nombre total de clients dans le cluster ;
# - anomalies : nombre de clients détectés comme anomalies dans le cluster.
anom_breakdown = df_active.groupby(
    ["cluster_id", "cluster_name"]
).agg(
    total=("anomaly_flag", "size"),
    anomalies=("anomaly_flag", "sum"),
)


# ------------------------------------------------------------
# 6) Calcul du pourcentage d’anomalies par cluster
# ------------------------------------------------------------

# On calcule le pourcentage d’anomalies dans chaque cluster.
#
# Formule :
# anomalies / total * 100
anom_breakdown["pct_anomalies"] = (
    anom_breakdown["anomalies"] / anom_breakdown["total"] * 100
).round(2)


# ------------------------------------------------------------
# 7) Affichage du tableau final
# ------------------------------------------------------------

# Ce tableau permet de voir quels clusters contiennent
# le plus de comportements atypiques.
anom_breakdown

Anomalies détectées : 277 (3.01%)

Anomalies par cluster :


### 11.1 Visualisation des anomalies

- Cette étape permet de visualiser les clients détectés comme anomalies par Isolation Forest.

- Deux visualisations sont utilisées :

1. **Distribution du `anomaly_score`**
   - Le score d’anomalie mesure à quel point un client est atypique.
   - Plus le score est élevé, plus le client est différent des autres.
   - Les anomalies doivent se situer dans la partie extrême de la distribution.

2. **Projection PCA 2D**
   - Les clients sont projetés dans un espace à deux dimensions.
   - Les clients normaux sont affichés en bleu.
   - Les anomalies sont affichées en rouge.
   - Les anomalies devraient se trouver plutôt aux frontières des clusters ou isolées en périphérie.

- Cette étape ne recalcule pas les anomalies.
- Elle sert uniquement à vérifier visuellement si la détection est cohérente.

- À la fin, on affiche aussi les 5 clients les plus atypiques selon `anomaly_score`.

In [34]:
# ------------------------------------------------------------
# 11.1 Visualisation des anomalies
# ------------------------------------------------------------

# On crée une figure avec deux graphiques côte à côte.
#
# 1 ligne, 2 colonnes :
# - axes[0] : histogramme des scores d’anomalie ;
# - axes[1] : projection PCA 2D avec anomalies.
#
# figsize=(14, 5) définit la taille globale de la figure.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))


# ------------------------------------------------------------
# 1) Récupération des scores et des flags d’anomalie
# ------------------------------------------------------------

# On récupère les scores d’anomalie sous forme de tableau numpy.
#
# anomaly_score mesure le degré d’atypie du client.
# Plus anomaly_score est élevé, plus le client est atypique.
scores = df_active["anomaly_score"].to_numpy()


# On récupère les flags d’anomalie.
#
# anomaly_flag = 1 signifie que le client est une anomalie.
# anomaly_flag = 0 signifie que le client est considéré comme normal.
flags = df_active["anomaly_flag"].to_numpy()


# ------------------------------------------------------------
# 2) Calcul du seuil d’anomalie
# ------------------------------------------------------------

# On cherche le plus petit score parmi les clients détectés comme anomalies.
#
# Cela donne le seuil à partir duquel les clients ont été classés comme anomalies.
#
# Exemple :
# si threshold = 0.62,
# alors les clients avec un anomaly_score supérieur ou égal à environ 0.62
# sont les plus atypiques.
#
# Si aucune anomalie n’est détectée, on met threshold = None.
threshold = scores[flags == 1].min() if flags.sum() > 0 else None


# ------------------------------------------------------------
# 3) Histogramme des scores des clients normaux
# ------------------------------------------------------------

# On affiche la distribution des scores pour les clients normaux.
#
# scores[flags == 0] sélectionne uniquement les clients non anomalies.
axes[0].hist(
    scores[flags == 0],
    bins=60,
    color="steelblue",
    edgecolor="white",
    label=f"Normaux (n={int((flags == 0).sum()):,})",
    alpha=0.8
)


# ------------------------------------------------------------
# 4) Histogramme des scores des anomalies
# ------------------------------------------------------------

# On affiche la distribution des scores pour les clients anomalies.
#
# scores[flags == 1] sélectionne uniquement les clients détectés comme anomalies.
#
# Les anomalies sont affichées en rouge pour les distinguer visuellement.
axes[0].hist(
    scores[flags == 1],
    bins=60,
    color="crimson",
    edgecolor="white",
    label=f"Anomalies (n={int(flags.sum()):,})",
    alpha=0.9
)


# ------------------------------------------------------------
# 5) Affichage du seuil d’anomalie
# ------------------------------------------------------------

# Si un seuil existe, on l’affiche avec une ligne verticale noire.
#
# Cette ligne permet de voir à partir de quel score
# les clients sont considérés comme anomalies.
if threshold is not None:
    axes[0].axvline(
        threshold,
        color="black",
        linestyle="--",
        linewidth=1,
        label=f"Seuil = {threshold:.3f}"
    )


# Nom de l’axe horizontal.
# Il indique que plus le score est élevé, plus le client est atypique.
axes[0].set_xlabel("anomaly_score (plus haut = plus atypique)")


# Nom de l’axe vertical.
# Il représente le nombre de clients dans chaque intervalle de score.
axes[0].set_ylabel("Nombre de clients")


# Titre du premier graphique.
axes[0].set_title("Distribution du score d'anomalie")


# Ajout de la légende.
axes[0].legend(fontsize=9)


# ------------------------------------------------------------
# 6) Préparation de l’échantillon pour la PCA
# ------------------------------------------------------------

# On limite le nombre de points à afficher pour éviter un graphique trop lourd.
#
# Si X contient plus de 40 000 clients, on prend seulement 40 000 clients.
# Sinon, on prend tous les clients.
sample_n = min(40_000, len(X))


# On tire aléatoirement les indices des clients à afficher.
#
# replace=False signifie qu’un même client ne peut pas être sélectionné deux fois.
idx_viz = rng.choice(
    len(X),
    size=sample_n,
    replace=False
)


# ------------------------------------------------------------
# 7) Projection PCA des clients sélectionnés
# ------------------------------------------------------------

# On projette les clients sélectionnés dans le plan PCA 2D.
#
# pca.transform utilise la PCA déjà entraînée précédemment.
# Le résultat X_viz contient deux coordonnées par client :
# - PC1 ;
# - PC2.
X_viz = pca.transform(X[idx_viz])


# On récupère les flags d’anomalie pour les clients sélectionnés.
flags_viz = flags[idx_viz]


# ------------------------------------------------------------
# 8) Affichage des clients normaux dans le plan PCA
# ------------------------------------------------------------

# On affiche les clients normaux en bleu.
#
# X_viz[flags_viz == 0, 0] correspond à l’axe PC1.
# X_viz[flags_viz == 0, 1] correspond à l’axe PC2.
#
# s=4 donne des petits points.
# alpha=0.25 rend les points transparents pour mieux voir les zones denses.
axes[1].scatter(
    X_viz[flags_viz == 0, 0],
    X_viz[flags_viz == 0, 1],
    s=4,
    alpha=0.25,
    color="steelblue",
    label="Normaux"
)


# ------------------------------------------------------------
# 9) Affichage des anomalies dans le plan PCA
# ------------------------------------------------------------

# On affiche les anomalies en rouge.
#
# Elles sont plus visibles grâce à :
# - une taille de point plus grande ;
# - une opacité plus forte ;
# - une bordure noire.
axes[1].scatter(
    X_viz[flags_viz == 1, 0],
    X_viz[flags_viz == 1, 1],
    s=10,
    alpha=0.9,
    color="crimson",
    label="Anomalies",
    edgecolor="black",
    linewidth=0.3
)


# Nom de l’axe X.
# On affiche aussi le pourcentage de variance expliqué par PC1.
axes[1].set_xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]:.1%})"
)


# Nom de l’axe Y.
# On affiche aussi le pourcentage de variance expliqué par PC2.
axes[1].set_ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]:.1%})"
)


# Titre du second graphique.
axes[1].set_title("Anomalies dans le plan PCA 2D")


# Ajout de la légende.
axes[1].legend(fontsize=9, markerscale=2)


# ------------------------------------------------------------
# 10) Affichage final des deux graphiques
# ------------------------------------------------------------

# Ajustement automatique de la mise en page.
plt.tight_layout()


# Affichage des graphiques.
plt.show()


# ------------------------------------------------------------
# 11) Affichage des 5 clients les plus atypiques
# ------------------------------------------------------------

# On affiche un message.
print(f"\nTop 5 clients les plus atypiques :")


# On sélectionne les 5 clients avec les plus grands anomaly_score.
#
# nlargest(5, "anomaly_score") trie les clients du plus atypique
# au moins atypique et garde les 5 premiers.
top_anom = df_active.nlargest(
    5,
    "anomaly_score"
)[
    [
        "MSISDN",
        "cluster_name",
        "risk_label",
        "AVG_CREDIT_AMOUNT",
        "TOTAL_OUTSTANDING_AMOUNT",
        "NB_SOS",
        "reimburse_ratio",
        "anomaly_score"
    ]
].reset_index(drop=True)


# Affichage du tableau des 5 clients les plus atypiques.
top_anom


Top 5 clients les plus atypiques :


### 11.2 Fusion ACTIVE (ML) + non-ACTIVE (déterministe)

- Cette étape reconstruit le dataset final complet.

- Le clustering ML et la détection d’anomalies ont été appliqués uniquement aux clients `ACTIVE`.

- Les clients non-`ACTIVE` ne sont pas soumis au clustering ML, car leur état administratif porte déjà une information forte.

- Pour les clients non-`ACTIVE`, on applique des règles déterministes :
  - `cluster_id = -1` : client non soumis au clustering ML ;
  - `cluster_name = STATE` : le nom du segment correspond à son état administratif ;
  - `anomaly_flag = 0` : pas d’anomalie calculée ;
  - `anomaly_score = 0.0` : score d’anomalie nul.

- Tous les clients, actifs et non actifs, gardent les informations de risque calculées précédemment :
  - `risk_score_raw` ;
  - `risk_score_final` ;
  - `risk_level` ;
  - `risk_label`.

- L’objectif est d’obtenir un dataset final unique contenant :
  - les informations métier du client ;
  - la segmentation comportementale ;
  - le niveau de risque ;
  - l’indicateur d’anomalie.

- Ce dataset final pourra ensuite être exporté et utilisé dans la plateforme décisionnelle ou par l' agent.

In [35]:
# ------------------------------------------------------------
# 11.2 Fusion ACTIVE (ML) + non-ACTIVE (déterministe)
# ------------------------------------------------------------

# On sélectionne tous les clients qui ne sont pas ACTIVE.
#
# Ces clients n'ont pas été utilisés dans le clustering ML,
# car leur STATE donne déjà une information métier importante.
#
# copy() permet de créer une copie indépendante.
# reset_index(drop=True) remet l'index à zéro.
df_other = df[df["STATE"] != "ACTIVE"].copy().reset_index(drop=True)


# ------------------------------------------------------------
# 1) Attribution d'un cluster déterministe aux non-ACTIVE
# ------------------------------------------------------------

# Les clients non-ACTIVE ne sont pas passés dans le clustering ML.
#
# On leur attribue donc cluster_id = -1.
# C'est une convention qui signifie :
# "non soumis au clustering ML".
df_other["cluster_id"] = -1


# Pour les clients non-ACTIVE, le nom du cluster devient simplement leur STATE.
#
# Exemple :
# - un client SUSPENDED aura cluster_name = "SUSPENDED" ;
# - un client ON-HOLD aura cluster_name = "ON-HOLD" ;
# - un client DISCONNECTED aura cluster_name = "DISCONNECTED".
df_other["cluster_name"] = df_other["STATE"]


# ------------------------------------------------------------
# 2) Gestion des anomalies pour les non-ACTIVE
# ------------------------------------------------------------

# La détection d'anomalies a été calculée uniquement sur les clients ACTIVE.
#
# Pour les clients non-ACTIVE, on ne calcule pas d'anomalie,
# car leur état administratif est déjà l'information principale.
#
# anomaly_flag = 0 signifie : pas d'anomalie calculée.
df_other["anomaly_flag"] = 0


# anomaly_score = 0.0 signifie qu'il n'y a pas de score d'anomalie
# calculé pour ces clients.
df_other["anomaly_score"] = 0.0


# ------------------------------------------------------------
# 3) Définition des colonnes à conserver
# ------------------------------------------------------------

# On définit la liste des colonnes qui seront gardées
# dans le dataset final.
#
# Cette liste contient :
# - les identifiants client ;
# - les variables métier ;
# - les flags comportementaux ;
# - les résultats du clustering ;
# - les scores et labels de risque ;
# - les informations d'anomalie.
KEEP_COLS = [
    "MSISDN",
    "STATE",

    "AVG_CREDIT_AMOUNT",
    "reimburse_ratio",
    "AVG_DAYS_SINCE_CREDIT",
    "TOTAL_OUTSTANDING_AMOUNT",
    "NB_SOS",

    "debt_to_credit",
    "credit_intensity",
    "tenure_days",

    "has_debt",
    "uses_sos",
    "never_repaid",
    "full_repayer",
    "is_dormant_like",

    "cluster_id",
    "cluster_name",

    "risk_score_raw",
    "risk_score_final",
    "risk_level",
    "risk_label",

    "anomaly_flag",
    "anomaly_score",
]


# ------------------------------------------------------------
# 4) Fusion des clients ACTIVE et non-ACTIVE
# ------------------------------------------------------------

# On fusionne :
# - df_active : clients ACTIVE avec clustering ML et anomalies ;
# - df_other  : clients non-ACTIVE avec règles déterministes.
#
# ignore_index=True permet de recréer un index propre dans final_df.
final_df = pd.concat(
    [
        df_active[KEEP_COLS],
        df_other[KEEP_COLS]
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 5) Affichage de la taille du dataset final
# ------------------------------------------------------------

# On affiche le nombre de lignes et de colonnes du dataset final.
#
# len(final_df) donne le nombre de clients.
# final_df.shape[1] donne le nombre de colonnes.
print(
    f"Dataset final : {len(final_df):,} lignes x {final_df.shape[1]} colonnes"
)


# ------------------------------------------------------------
# 6) Résumé de la segmentation comportementale
# ------------------------------------------------------------

# On affiche un titre.
print("\nSegmentation comportementale (cluster_name) :")


# On regroupe le dataset final par cluster_id et cluster_name.
#
# size() compte le nombre de clients dans chaque segment.
# reset_index(name="count") transforme le résultat en DataFrame.
seg = final_df.groupby(
    ["cluster_id", "cluster_name"]
).size().reset_index(name="count")


# On calcule le pourcentage de chaque segment
# par rapport au nombre total de clients.
seg["pct"] = (
    seg["count"] / len(final_df) * 100
).round(2)


# On affiche les segments triés du plus grand au plus petit.
print(
    seg.sort_values(
        "count",
        ascending=False
    ).to_string(index=False)
)


# ------------------------------------------------------------
# 7) Résumé de la Risk Ladder
# ------------------------------------------------------------

# On affiche un titre.
print("\nRisk ladder (risk_level) :")


# On regroupe les clients par niveau et label de risque.
#
# Exemple :
# - niveau 1 : Sans risque ;
# - niveau 2 : Normal ;
# - niveau 3 : À risque ;
# - niveau 4 : Haut risque ;
# - niveau 5 : Blacklist.
lad = final_df.groupby(
    ["risk_level", "risk_label"]
).size().reset_index(name="count")


# On calcule le pourcentage de chaque niveau de risque.
lad["pct"] = (
    lad["count"] / len(final_df) * 100
).round(2)


# On affiche le tableau final de la Risk Ladder.
print(
    lad.to_string(index=False)
)

Dataset final : 9,748 lignes x 23 colonnes

Segmentation comportementale (cluster_name) :
 cluster_id cluster_name  count    pct
          0     Standard   6602 67.730
          1   Bon-payeur   2606 26.730
         -1    SUSPENDED    497  5.100
         -1 DISCONNECTED     28  0.290
         -1      ON-HOLD     15  0.150

Risk ladder (risk_level) :
 risk_level  risk_label  count    pct
          1 Sans risque   2241 22.990
          2      Normal   3723 38.190
          3    A risque   2105 21.590
          4 Haut risque    702  7.200
          5   Blacklist    977 10.020


### 11.3 Dashboard final — segmentation croisée

- Cette étape construit une vue finale croisée entre :
  - `cluster_name` : segment comportemental du client ;
  - `risk_label` : niveau de risque final ;
  - `anomaly_flag` : indicateur d’anomalie.

- L’objectif est de résumer les résultats finaux dans un tableau exploitable.

- La première heatmap affiche les effectifs :
  - combien de clients se trouvent dans chaque couple `cluster_name × risk_label`.

- La deuxième heatmap affiche les pourcentages par cluster :
  - pour chaque cluster, on observe la répartition des clients entre les niveaux de risque.

- Cette visualisation permet de vérifier la cohérence globale :
  - les `Bon-payeur` doivent surtout être en `Sans risque` ou `Normal` ;
  - les `Standard` peuvent se répartir entre `Normal`, `À risque` et `Haut risque` ;
  - les clients `SUSPENDED` et `DISCONNECTED` doivent être en `Blacklist` ;
  - les clients `ON-HOLD` doivent être en `Haut risque`.

- À la fin, un tableau supplémentaire affiche les anomalies par couple `cluster_name × risk_label`.

- Cette vue finale est utile pour l'agent, car elle permet de router les actions selon le profil comportemental, le risque et l’anomalie.

In [36]:
# ------------------------------------------------------------
# 11.3 Dashboard final — segmentation croisée
# ------------------------------------------------------------

# Ordre souhaité des niveaux de risque dans les tableaux.
#
# Cela permet d'afficher les colonnes dans l'ordre logique :
# du moins risqué au plus risqué.
LABEL_ORDER = [
    "Sans risque",
    "Normal",
    "A risque",
    "Haut risque",
    "Blacklist"
]


# Ordre souhaité des clusters / segments comportementaux.
#
# On place d'abord les clusters ML des clients ACTIVE :
# - Bon-payeur
# - Standard
#
# Puis les segments déterministes des non-ACTIVE :
# - ON-HOLD
# - SUSPENDED
# - DISCONNECTED
# - OTHER
CLUSTER_ORDER = [
    "Bon-payeur",
    "Standard",
    "ON-HOLD",
    "SUSPENDED",
    "DISCONNECTED",
    "OTHER"
]


# ------------------------------------------------------------
# 1) Création du tableau croisé cluster_name × risk_label
# ------------------------------------------------------------

# On groupe le dataset final par :
# - cluster_name : segment comportemental ;
# - risk_label : niveau de risque.
#
# size() compte le nombre de clients dans chaque couple.
#
# unstack(fill_value=0) transforme risk_label en colonnes.
# Les cellules vides sont remplacées par 0.
pivot = (
    final_df
    .groupby(["cluster_name", "risk_label"])
    .size()
    .unstack(fill_value=0)
)


# ------------------------------------------------------------
# 2) Réorganisation des lignes et colonnes
# ------------------------------------------------------------

# On réordonne les lignes selon CLUSTER_ORDER.
# On garde seulement les clusters réellement présents dans le tableau.
#
# On réordonne aussi les colonnes selon LABEL_ORDER.
# On garde seulement les labels présents.
#
# fill_value=0 remplace les valeurs absentes par 0.
pivot = pivot.reindex(
    index=[c for c in CLUSTER_ORDER if c in pivot.index],
    columns=[l for l in LABEL_ORDER if l in pivot.columns],
    fill_value=0
)


# ------------------------------------------------------------
# 3) Affichage du tableau en effectifs
# ------------------------------------------------------------

# On affiche un message explicatif.
print("Pivot cluster_name x risk_label (nombre de clients) :")


# On affiche le tableau sous forme texte.
#
# Chaque cellule représente le nombre de clients
# dans un couple cluster_name × risk_label.
print(pivot.to_string())


# ------------------------------------------------------------
# 4) Création de la figure avec deux heatmaps
# ------------------------------------------------------------

# On crée une figure avec deux graphiques côte à côte.
#
# axes[0] : heatmap des effectifs.
# axes[1] : heatmap des pourcentages par cluster.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))


# ------------------------------------------------------------
# 5) Heatmap des effectifs
# ------------------------------------------------------------

# Cette heatmap affiche le nombre de clients dans chaque couple :
# cluster_name × risk_label.
#
# annot=True affiche les valeurs dans les cases.
# fmt="d" indique que les valeurs sont des entiers.
# cmap="YlOrRd" utilise une palette jaune-orange-rouge.
sns.heatmap(
    pivot,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    cbar_kws={"label": "Nombre de clients"},
    ax=axes[0]
)


# Titre et labels du premier graphique.
axes[0].set_title("Segmentation croisee — effectifs")
axes[0].set_xlabel("Risk ladder")
axes[0].set_ylabel("Cluster comportemental")


# ------------------------------------------------------------
# 6) Calcul des pourcentages par cluster
# ------------------------------------------------------------

# On transforme les effectifs en pourcentages par ligne.
#
# pivot.sum(axis=1) calcule le total de chaque cluster.
# div(..., axis=0) divise chaque ligne par son total.
# * 100 transforme la proportion en pourcentage.
pivot_pct = pivot.div(
    pivot.sum(axis=1),
    axis=0
) * 100


# ------------------------------------------------------------
# 7) Heatmap des pourcentages par cluster
# ------------------------------------------------------------

# Cette heatmap affiche la distribution du risque à l'intérieur de chaque cluster.
#
# Exemple :
# Pour le cluster Bon-payeur, on regarde quel pourcentage est :
# - Sans risque ;
# - Normal ;
# - À risque ;
# - Haut risque ;
# - Blacklist.
sns.heatmap(
    pivot_pct,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    cbar_kws={"label": "% par ligne (cluster)"},
    ax=axes[1]
)


# Titre et labels du deuxième graphique.
axes[1].set_title("Distribution du risque DANS chaque cluster (%)")
axes[1].set_xlabel("Risk ladder")
axes[1].set_ylabel("Cluster comportemental")


# Ajustement automatique de la mise en page.
plt.tight_layout()


# Affichage des deux graphiques.
plt.show()


# ------------------------------------------------------------
# 8) Analyse des anomalies par couple cluster × risque
# ------------------------------------------------------------

# On affiche un message.
print("\nAnomalies par couple cluster x risk :")


# On sélectionne uniquement les clients détectés comme anomalies.
#
# anomaly_flag = 1 signifie que le client est atypique.
#
# Ensuite, on groupe par :
# - cluster_name ;
# - risk_label.
#
# Le but est de savoir où se trouvent les anomalies :
# dans quel cluster et dans quel niveau de risque.
anom_pivot = (
    final_df[final_df["anomaly_flag"] == 1]
    .groupby(["cluster_name", "risk_label"])
    .size()
    .unstack(fill_value=0)
)


# Si le tableau des anomalies n'est pas vide,
# on le réorganise selon le même ordre que les heatmaps.
if not anom_pivot.empty:

    anom_pivot = anom_pivot.reindex(
        index=[c for c in CLUSTER_ORDER if c in anom_pivot.index],
        columns=[l for l in LABEL_ORDER if l in anom_pivot.columns],
        fill_value=0
    )

    # Affichage du tableau des anomalies.
    print(anom_pivot.to_string())

Pivot cluster_name x risk_label (nombre de clients) :
risk_label    Sans risque  Normal  A risque  Haut risque  Blacklist
cluster_name                                                       
Bon-payeur            653    1086       543          198        126
Standard             1588    2637      1562          489        326
ON-HOLD                 0       0         0           15          0
SUSPENDED               0       0         0            0        497
DISCONNECTED            0       0         0            0         28

Anomalies par couple cluster x risk :
risk_label    Sans risque  Normal  A risque  Haut risque  Blacklist
cluster_name                                                       
Bon-payeur              9       6         5            5          0
Standard               61      98        59           22         12


## 12. Contrat de sortie pour l’agent décisionnel

Le CSV final doit être directement exploitable par l’agent décisionnel, sans transformation supplémentaire.

L’objectif de cette section est de préparer un fichier final aligné avec le format attendu par le backend et par le graphe décisionnel.

L’agent décisionnel attend deux grands blocs d’informations :

1. `features` : les variables métier brutes utilisées pour comprendre le profil du client ;
2. `ml_outputs` : les résultats issus du notebook ML, comme le cluster, le niveau de risque, le score final, l’anomalie et les facteurs explicatifs.

| Clé attendue                        | Type                    | Source notebook               | Transformation                   |
| ----------------------------------- | ----------------------- | ----------------------------- | -------------------------------- |
| `msisdn`                            | str                     | `MSISDN`                      | renommage en minuscule           |
| `cluster_id`                        | int                     | `cluster_id`                  | direct                           |
| `risk_tier`                         | `low`, `medium`, `high` | `risk_label`                  | mapping 5 niveaux vers 3 niveaux |
| `final_risk_score`                  | float                   | `risk_score_final`            | renommage                        |
| `is_anomaly`                        | bool                    | `anomaly_flag`                | conversion en booléen            |
| `top_drivers`                       | liste JSON              | calculé à partir des features | z-scores + JSON                  |
| `features.TOTAL_OUTSTANDING_AMOUNT` | float                   | `TOTAL_OUTSTANDING_AMOUNT`    | direct                           |
| `features.AVG_REIMBURSE_RATIO`      | float                   | `reimburse_ratio`             | renommage                        |
| `features.NB_SOS`                   | float                   | `NB_SOS`                      | direct                           |

Le Risk Ladder initial contient 5 niveaux :

* `Sans risque`
* `Normal`
* `A risque`
* `Haut risque`
* `Blacklist`

Pour l’agent décisionnel, ces 5 niveaux sont regroupés en 3 tiers :

* `Sans risque` et `Normal` deviennent `low`
* `A risque` devient `medium`
* `Haut risque` et `Blacklist` deviennent `high`

On conserve également `risk_level` et `risk_label` dans le CSV final afin de ne pas perdre l’information détaillée en 5 niveaux.

Cette section permet donc de transformer le dataset final ML en un fichier prêt à être consommé par l’agent décisionnel.


### 12.1 Mapping 5 niveaux -> 3 tiers

In [37]:
# ------------------------------------------------------------
# 12.1 Mapping 5 niveaux -> 3 tiers + renommage agent décisionnel
# ------------------------------------------------------------

# Dictionnaire de mapping entre les 5 niveaux de la Risk Ladder
# et les 3 niveaux attendus par l’agent décisionnel.
RISK_TIER_MAP = {
    "Sans risque": "low",      # risque faible
    "Normal":      "low",      # risque faible / suivi routine
    "A risque":    "medium",   # risque moyen
    "Haut risque": "high",     # risque élevé
    "Blacklist":   "high",     # risque maximal
}

# Création de la colonne risk_tier.
# Elle transforme risk_label en low / medium / high.
final_df["risk_tier"] = final_df["risk_label"].map(RISK_TIER_MAP)

# Renommage logique du score final pour l’agent.
# L’agent attend une clé appelée final_risk_score.
final_df["final_risk_score"] = final_df["risk_score_final"]

# Conversion du flag d’anomalie en booléen.
# 0 devient False, 1 devient True.
final_df["is_anomaly"] = final_df["anomaly_flag"].astype(bool)

# Création de la colonne attendue AVG_REIMBURSE_RATIO
# à partir de la colonne existante reimburse_ratio.
final_df["AVG_REIMBURSE_RATIO"] = final_df["reimburse_ratio"]

# Affichage de la distribution des 3 tiers.
print("Distribution risk_tier (3 niveaux agent décisionnel) :")

# Comptage du nombre de clients par tier.
tier_dist = final_df["risk_tier"].value_counts().reindex(["low", "medium", "high"])

# Calcul du pourcentage par tier.
tier_pct = (tier_dist / len(final_df) * 100).round(2)

# Affichage sous forme de tableau.
print(pd.DataFrame({"count": tier_dist, "pct": tier_pct}).to_string())

# Vérification de la correspondance entre risk_label et risk_tier.
print("\nCorrespondance risk_label -> risk_tier :")
print(pd.crosstab(final_df["risk_label"], final_df["risk_tier"]).to_string())

Distribution risk_tier (3 niveaux agent décisionnel) :
           count    pct
risk_tier              
low         5964 61.180
medium      2105 21.590
high        1679 17.220

Correspondance risk_label -> risk_tier :
risk_tier    high   low  medium
risk_label                     
A risque        0     0    2105
Blacklist     977     0       0
Haut risque   702     0       0
Normal          0  3723       0
Sans risque     0  2241       0


### 12.2 Calcul de `top_drivers` par client

* Cette étape sert à expliquer les décisions de l’agent décisionnel.

* Le `risk_tier` indique le niveau de risque du client, mais il ne dit pas pourquoi le client est classé ainsi.

* Pour expliquer chaque client, on calcule les variables qui s’éloignent le plus de la moyenne des clients `ACTIVE`.

* On utilise pour cela un `z_score`.

* Un `z_score` mesure l’écart entre la valeur du client et la moyenne de référence.

* La population `ACTIVE` est utilisée comme référence métier.

* Pour chaque client, on calcule les z-scores sur 9 variables clés.

* Ensuite, on garde les 4 variables avec les plus grands écarts en valeur absolue `|z_score|`.

* Ces 4 variables deviennent les `top_drivers`.

* Les `top_drivers` ne recalculent pas le risque : ils servent uniquement à expliquer les facteurs majeurs du client.

* Le résultat est stocké dans la colonne `top_drivers` sous forme de texte JSON, afin d’être exporté dans le CSV et relu par l’agent avec `json.loads`.


In [38]:
import json as _json

AGENT_FEATURES = [
    "AVG_CREDIT_AMOUNT",
    "TOTAL_OUTSTANDING_AMOUNT",
    "NB_SOS",
    "reimburse_ratio",
    "debt_to_credit",
    "credit_intensity",
    "has_debt",
    "never_repaid",
    "full_repayer",
]

pop_active = df_active[AGENT_FEATURES].astype(float)
mu_pop = pop_active.mean()
sigma_pop = pop_active.std().replace(0, 1.0)

print("Population de reference (ACTIVE) :")
print(f"  n = {len(pop_active):,}")
print(pd.DataFrame({"mean": mu_pop.round(3), "std": sigma_pop.round(3)}).to_string())

# Version vectorisée : plus rapide et stable qu'un apply(axis=1).
vals_df = final_df[AGENT_FEATURES].astype(float)
z_df = (vals_df - mu_pop) / sigma_pop
feature_names = np.array(AGENT_FEATURES)
vals_matrix = vals_df.to_numpy()
z_matrix = z_df.to_numpy()

top_drivers = []
for vals_row, z_row in zip(vals_matrix, z_matrix):
    order = np.argsort(np.abs(z_row))[::-1][:4]
    drivers = [
        {
            "feature": str(feature_names[j]),
            "value": round(float(vals_row[j]), 3),
            "z_score": round(float(z_row[j]), 3),
        }
        for j in order
    ]
    top_drivers.append(_json.dumps(drivers, ensure_ascii=False))

final_df["top_drivers"] = top_drivers

print("\nExemple pour le 1er client :")
print(f"  msisdn      = {final_df['MSISDN'].iloc[0]}")
print(f"  risk_tier   = {final_df['risk_tier'].iloc[0]}")
print(f"  top_drivers = {final_df['top_drivers'].iloc[0]}")


Population de reference (ACTIVE) :
  n = 9,208
                           mean    std
AVG_CREDIT_AMOUNT         2.148  2.205
TOTAL_OUTSTANDING_AMOUNT  5.220  7.672
NB_SOS                   17.203 23.906
reimburse_ratio           0.796  0.240
debt_to_credit            2.909  4.064
credit_intensity          0.634  1.222
has_debt                  0.701  0.458
never_repaid              0.036  0.187
full_repayer              0.346  0.476

Exemple pour le 1er client :
  msisdn      = 21696330926
  risk_tier   = low
  top_drivers = [{"feature": "full_repayer", "value": 0.0, "z_score": -0.728}, {"feature": "has_debt", "value": 1.0, "z_score": 0.652}, {"feature": "debt_to_credit", "value": 0.315, "z_score": -0.638}, {"feature": "TOTAL_OUTSTANDING_AMOUNT", "value": 0.8, "z_score": -0.576}]


### 12.3 Visualisation du contrat agent décisionnel

* Cette étape permet de vérifier visuellement que la sortie finale est cohérente avant l’export CSV.

* On visualise les informations que l’agent décisionnel va utiliser.

* Trois vues sont produites :

1. **Distribution de `risk_tier`**

   * Montre la proportion des clients en `low`, `medium` et `high`.
   * C’est la vision simplifiée que l’agent décisionnel utilisera.

2. **Mapping `risk_label` vers `risk_tier`**

   * Vérifie la correspondance entre les 5 niveaux de la Risk Ladder et les 3 tiers agent.
   * `Sans risque` et `Normal` vont vers `low`.
   * `A risque` va vers `medium`.
   * `Haut risque` et `Blacklist` vont vers `high`.

3. **Taux d’anomalies par `risk_tier`**

   * Vérifie si les anomalies sont plus concentrées dans les tiers élevés.
   * On s’attend à voir plus d’anomalies dans `high` que dans `low`.

* Cette étape ne modifie pas les données.
* Elle sert uniquement à valider la cohérence du contrat de sortie destiné à l’agent décisionnel.


In [39]:

# ------------------------------------------------------------
# 12.3 Visualisation du contrat agent décisionnel
# ------------------------------------------------------------

# On crée une figure avec 3 graphiques côte à côte.
# axes[0] : camembert risk_tier
# axes[1] : mapping risk_label -> risk_tier
# axes[2] : taux d'anomalies par risk_tier
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))


# ------------------------------------------------------------
# 1) Définition des couleurs des 3 tiers
# ------------------------------------------------------------

# Chaque tier reçoit une couleur métier :
# low    -> vert
# medium -> jaune
# high   -> rouge
tier_colors = {
    "low": "#4CAF50",
    "medium": "#FFC107",
    "high": "#F44336"
}


# ------------------------------------------------------------
# 2) Distribution des clients par risk_tier
# ------------------------------------------------------------

# On compte le nombre de clients dans chaque tier.
# reindex impose l'ordre low, medium, high.
# fillna(0) évite les valeurs manquantes si un tier est absent.
tier_dist = (
    final_df["risk_tier"]
    .value_counts()
    .reindex(["low", "medium", "high"])
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 3) Graphique 1 : camembert risk_tier
# ------------------------------------------------------------

# Ce camembert montre la proportion de clients que l'agent verra
# dans chaque catégorie : low, medium ou high.
axes[0].pie(
    tier_dist,
    labels=[f"{t}\n({n:,})" for t, n in tier_dist.items()],
    autopct="%1.1f%%",
    colors=[tier_colors[t] for t in tier_dist.index],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)

# Titre du premier graphique.
axes[0].set_title("Distribution risk_tier (vue agent décisionnel)")


# ------------------------------------------------------------
# 4) Tableau croisé risk_label -> risk_tier
# ------------------------------------------------------------

# On crée un tableau croisé entre :
# - risk_label : les 5 niveaux de la Risk Ladder
# - risk_tier  : les 3 niveaux attendus par l'agent
cross = pd.crosstab(
    final_df["risk_label"],
    final_df["risk_tier"]
)

# On impose l'ordre logique des lignes et colonnes.
cross = cross.reindex(
    index=["Sans risque", "Normal", "A risque", "Haut risque", "Blacklist"],
    columns=["low", "medium", "high"],
    fill_value=0
)


# ------------------------------------------------------------
# 5) Graphique 2 : mapping 5 niveaux -> 3 tiers
# ------------------------------------------------------------

# Ce graphique en barres empilées montre comment les 5 niveaux
# sont regroupés dans les 3 tiers de l'agent décisionnel.
cross.plot(
    kind="barh",
    stacked=True,
    ax=axes[1],
    color=[tier_colors[t] for t in cross.columns],
    edgecolor="white",
)

# Titres et labels du deuxième graphique.
axes[1].set_title("Mapping risk_label (5) -> risk_tier (3)")
axes[1].set_xlabel("Nombre de clients")
axes[1].set_ylabel("")
axes[1].legend(title="risk_tier", loc="lower right")


# ------------------------------------------------------------
# 6) Calcul du taux d'anomalies par tier
# ------------------------------------------------------------

# On calcule la moyenne de is_anomaly pour chaque risk_tier.
# Comme is_anomaly vaut True/False, la moyenne donne directement
# le pourcentage d'anomalies.
anom_rate = (
    final_df
    .groupby("risk_tier")["is_anomaly"]
    .mean()
    .mul(100)
    .reindex(["low", "medium", "high"])
    .fillna(0)
)


# ------------------------------------------------------------
# 7) Graphique 3 : taux d'anomalies par risk_tier
# ------------------------------------------------------------

# Ce graphique montre le pourcentage d'anomalies dans chaque tier.
# On vérifie que les tiers plus risqués concentrent davantage d'anomalies.
axes[2].bar(
    anom_rate.index,
    anom_rate.values,
    color=[tier_colors[t] for t in anom_rate.index],
    edgecolor="white",
    linewidth=1.5,
)

# On ajoute le pourcentage au-dessus de chaque barre.
for i, v in enumerate(anom_rate.values):
    axes[2].text(
        i,
        v + 0.1,
        f"{v:.1f}%",
        ha="center",
        fontweight="bold"
    )

# Titres et labels du troisième graphique.
axes[2].set_ylabel("% anomalies dans le tier")
axes[2].set_title("Taux d'anomalies par risk_tier")

# On ajuste l'échelle de l'axe Y pour laisser de l'espace aux labels.
axes[2].set_ylim(0, max(anom_rate.max() * 1.25, 1))


# ------------------------------------------------------------
# 8) Affichage final
# ------------------------------------------------------------

# Ajustement automatique de la mise en page.
plt.tight_layout()

# Affichage des trois graphiques.
plt.show()



### 12.4 Export CSV final aligné sur le contrat de l’agent décisionnel

* Cette étape exporte le dataset final dans un fichier CSV.

* Le CSV doit être directement exploitable par l’agent décisionnel et le backend.

* On fixe d’abord la liste des colonnes à exporter avec `AGENT_EXPORT_COLS`.

* L’ordre des colonnes est organisé pour faciliter la lecture :

  * identité du client ;
  * segmentation comportementale ;
  * niveaux et scores de risque ;
  * anomalie ;
  * facteurs explicatifs ;
  * variables métier.

* La colonne `MSISDN` est renommée en `msisdn`, car l’agent attend une clé en minuscule.

* Avant l’export, le code vérifie que toutes les colonnes attendues existent.

* Si une colonne manque, le code arrête l’export avec une erreur claire.

* Le fichier final est exporté sous le nom `clients_segmented.csv`.

* L’encodage `utf-8-sig` est utilisé pour assurer une bonne compatibilité avec Excel.

* À la fin, le code affiche :

  * le chemin du fichier exporté ;
  * le nombre de lignes ;
  * le nombre de colonnes ;
  * le type de chaque colonne ;
  * un aperçu du fichier final.


In [40]:

# ============================================================
# 12.4 Export final compatible agent décisionnel / Backend Bad Debts
# ============================================================

# Liste des colonnes à exporter dans le CSV final.
#
# L’ordre est choisi pour améliorer la lisibilité :
# 1. identité du client ;
# 2. segmentation comportementale ;
# 3. niveaux et scores de risque ;
# 4. anomalie ;
# 5. facteurs explicatifs ;
# 6. variables métier.
AGENT_EXPORT_COLS = [
    "msisdn",
    "STATE",

    "cluster_id",
    "cluster_name",

    "risk_level",
    "risk_label",
    "risk_tier",

    "final_risk_score",
    "risk_score_raw",

    "is_anomaly",
    "anomaly_score",

    "top_drivers",

    "AVG_CREDIT_AMOUNT",
    "AVG_REIMBURSE_RATIO",
    "AVG_DAYS_SINCE_CREDIT",

    "TOTAL_OUTSTANDING_AMOUNT",
    "NB_SOS",

    "debt_to_credit",
    "credit_intensity",
    "tenure_days",

    "has_debt",
    "uses_sos",
    "never_repaid",
    "full_repayer",
    "is_dormant_like",
]


# ------------------------------------------------------------
# 1) Création du DataFrame à exporter
# ------------------------------------------------------------

# On crée une copie de final_df pour préparer l’export.
#
# La colonne MSISDN est renommée en msisdn,
# car l’agent décisionnel attend une clé en minuscule.
export_df = final_df.rename(
    columns={"MSISDN": "msisdn"}
).copy()


# ------------------------------------------------------------
# 2) Vérification des colonnes attendues
# ------------------------------------------------------------

# On vérifie que toutes les colonnes définies dans AGENT_EXPORT_COLS
# existent bien dans export_df.
#
# Si une colonne manque, elle sera ajoutée à la liste missing_cols.
missing_cols = [
    col for col in AGENT_EXPORT_COLS
    if col not in export_df.columns
]


# Si une ou plusieurs colonnes sont manquantes,
# on arrête le programme avec une erreur explicite.
#
# Cela évite d’exporter un fichier incomplet ou non compatible
# avec l’agent décisionnel.
if missing_cols:
    raise ValueError(
        "Colonnes manquantes pour l'export agent décisionnel : "
        + ", ".join(missing_cols)
    )


# ------------------------------------------------------------
# 3) Sélection des colonnes finales
# ------------------------------------------------------------

# On garde uniquement les colonnes nécessaires à l’agent décisionnel.
#
# Cela permet :
# - de supprimer les colonnes intermédiaires inutiles ;
# - de garantir l’ordre final des colonnes ;
# - de livrer un fichier propre et stable.
export_df = export_df[AGENT_EXPORT_COLS]


# ------------------------------------------------------------
# 4) Création du dossier d’export
# ------------------------------------------------------------

# On définit le dossier dans lequel le CSV final sera enregistré.
OUT_DIR = Path(os.environ.get('BAD_DEBTS_EXPORT_DIR', r'C:\\Users\\benab\\OneDrive\\Bureau\\PFE TT\\PLATEFORME TT\\backend\\runtime_data\\ml_imports\\28'))


# On crée le dossier s’il n’existe pas déjà.
#
# parents=True permet de créer les dossiers parents si nécessaire.
# exist_ok=True évite une erreur si le dossier existe déjà.
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Chemin complet du fichier CSV final.
OUT_PATH = Path(os.environ.get('BAD_DEBTS_CLIENTS_SEGMENTED_FILE', str(EXPORT_DIR / 'clients_segmented.csv')))


# ------------------------------------------------------------
# 5) Export du CSV final
# ------------------------------------------------------------

# Export du DataFrame en CSV.
#
# index=False évite d’ajouter l’index pandas comme colonne.
# encoding="utf-8-sig" permet une bonne compatibilité avec Excel.
export_df.to_csv(
    OUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 6) Résumé de l’export
# ------------------------------------------------------------

# Affichage d’un séparateur pour rendre la sortie plus lisible.
print("=" * 90)

# Message indiquant que l’export est compatible avec l’agent décisionnel.
print("Export final compatible agent décisionnel / Backend Bad Debts")

print("=" * 90)

# Affichage du chemin du fichier exporté.
print(f"Export OK : {OUT_PATH}")

# Affichage du nombre de lignes.
print(f"Lignes    : {len(export_df):,}")

# Affichage du nombre de colonnes.
print(f"Colonnes  : {export_df.shape[1]}")


# ------------------------------------------------------------
# 7) Vérification des types de colonnes
# ------------------------------------------------------------

# On affiche le type de chaque colonne exportée.
#
# On affiche aussi un exemple de valeur pour vérifier rapidement
# que le contenu est cohérent.
print("\nTypes des colonnes exportées :")

for col in AGENT_EXPORT_COLS:

    # On récupère un exemple de valeur dans la première ligne.
    # [:70] limite l’affichage à 70 caractères pour éviter les lignes trop longues.
    example_value = str(export_df[col].iloc[0])[:70]

    # Affichage :
    # nom de colonne, type pandas, exemple de valeur.
    print(
        f"  {col:30s} {str(export_df[col].dtype):12s} exemple: {example_value}"
    )


# ------------------------------------------------------------
# 8) Aperçu du fichier final
# ------------------------------------------------------------

# Affichage des premières lignes du CSV final.
print("\nAperçu export :")

display(export_df.head())



Export final compatible agent décisionnel / Backend Bad Debts
Export OK : C:\Users\benab\OneDrive\Bureau\PFE TT\PLATEFORME TT\backend\runtime_data\ml_imports\28\clients_segmented.csv
Lignes    : 9,748
Colonnes  : 25

Types des colonnes exportées :
  msisdn                         int64        exemple: 21696330926
  STATE                          object       exemple: ACTIVE
  cluster_id                     int64        exemple: 0
  cluster_name                   object       exemple: Standard
  risk_level                     int64        exemple: 2
  risk_label                     object       exemple: Normal
  risk_tier                      object       exemple: low
  final_risk_score               float64      exemple: 0.14583541335014946
  risk_score_raw                 float64      exemple: 0.20833630478592782
  is_anomaly                     bool         exemple: False
  anomaly_score                  float64      exemple: 0.4868688220343701
  top_drivers                    object

### 12.5 Sanity check — simulation d’une requête agent décisionnel

* Cette étape sert à vérifier que le CSV final est bien compatible avec l’agent décisionnel.

* On simule ce qui se passera lors d’un appel réel à l’agent.

* Le code prend une ligne au hasard dans `export_df`.

* À partir de cette ligne, il construit deux objets :

  * `features_payload` : contient les variables métier brutes attendues par le nœud de profiling ;
  * `ml_outputs_payload` : contient les résultats ML et risque attendus par l’agent décisionnel.

* Ensuite, le code vérifie que toutes les clés nécessaires sont présentes.

* Il vérifie aussi que `risk_tier` contient une valeur valide : `low`, `medium` ou `high`.

* Cette étape ne modifie pas les données.

* Elle sert uniquement à tester le contrat de sortie avant utilisation réelle par le backend ou l’agent décisionnel.


In [41]:

# ------------------------------------------------------------
# 12.5 Sanity check — simulation d'une requête agent décisionnel
# ------------------------------------------------------------

# On sélectionne un client au hasard dans le DataFrame exporté.
#
# export_df correspond au fichier final prêt à être exporté.
# sample(1) prend une seule ligne.
# random_state permet d'obtenir toujours le même exemple à chaque exécution.
# iloc[0] récupère la ligne sous forme de Series pandas.
sample = export_df.sample(1, random_state=RANDOM_STATE).iloc[0]


# ------------------------------------------------------------
# 1) Construction du payload features
# ------------------------------------------------------------

# features_payload contient les variables métier brutes.
#
# Ce sont les informations que le nœud de profiling utilise
# pour comprendre rapidement le profil du client.
#
# Chaque valeur est convertie en float pour garantir un format numérique propre.
features_payload = {
    "TOTAL_OUTSTANDING_AMOUNT": float(sample["TOTAL_OUTSTANDING_AMOUNT"]),
    "AVG_REIMBURSE_RATIO":      float(sample["AVG_REIMBURSE_RATIO"]),
    "NB_SOS":                   float(sample["NB_SOS"]),
}


# ------------------------------------------------------------
# 2) Construction du payload ml_outputs
# ------------------------------------------------------------

# ml_outputs_payload contient les résultats issus du notebook ML.
#
# Ces informations sont utilisées par l’agent décisionnel pour :
# - connaître le cluster du client ;
# - connaître son niveau de risque simplifié ;
# - connaître son score final ;
# - expliquer la décision avec top_drivers ;
# - savoir si le client est une anomalie.
ml_outputs_payload = {
    # Identifiant du cluster comportemental.
    "cluster_id": int(sample["cluster_id"]),

    # Niveau de risque simplifié attendu par l’agent :
    # low, medium ou high.
    "risk_tier": str(sample["risk_tier"]),

    # Score final de risque.
    "final_risk_score": float(sample["final_risk_score"]),

    # top_drivers est stocké sous forme de texte JSON dans le CSV.
    # json.loads permet de le reconvertir en liste Python.
    "top_drivers": _json.loads(sample["top_drivers"]),

    # Booléen indiquant si le client est détecté comme anomalie.
    "is_anomaly": bool(sample["is_anomaly"]),
}


# ------------------------------------------------------------
# 3) Affichage des informations du client simulé
# ------------------------------------------------------------

# On affiche l'identifiant client utilisé dans la simulation.
print(f"Simulation pour msisdn = {sample['msisdn']}")

# On affiche quelques informations générales pour vérifier le profil.
print(f"  STATE         = {sample['STATE']}")
print(f"  cluster_name  = {sample['cluster_name']}")
print(f"  risk_label    = {sample['risk_label']} (level {sample['risk_level']})")
print()


# ------------------------------------------------------------
# 4) Affichage du payload features
# ------------------------------------------------------------

print("features (attendu par profiling_node) :")

# On parcourt le dictionnaire features_payload.
# k = nom de la feature.
# v = valeur de la feature.
for k, v in features_payload.items():
    print(f"  {k:30s} = {v}")


# ------------------------------------------------------------
# 5) Affichage du payload ml_outputs
# ------------------------------------------------------------

print("\nml_outputs (attendu par profiling/explainability/decision/message) :")

# On parcourt les sorties ML.
for k, v in ml_outputs_payload.items():

    # Cas particulier : top_drivers est une liste de dictionnaires.
    # On l'affiche proprement ligne par ligne.
    if k == "top_drivers":
        print(f"  {k}:")

        # Chaque driver contient :
        # - feature : nom de la variable ;
        # - value : valeur réelle du client ;
        # - z_score : écart à la moyenne ACTIVE.
        for d in v:
            print(
                f"    - {d['feature']:26s} "
                f"value={d['value']:>10} "
                f"z={d['z_score']:+.2f}"
            )

    # Pour les autres champs, on affiche directement la valeur.
    else:
        print(f"  {k:18s} = {v}")


# ------------------------------------------------------------
# 6) Définition des clés attendues
# ------------------------------------------------------------

# Liste des features obligatoires attendues par l’agent.
REQUIRED_FEATURES = {
    "TOTAL_OUTSTANDING_AMOUNT",
    "AVG_REIMBURSE_RATIO",
    "NB_SOS"
}

# Liste des sorties ML obligatoires attendues par l’agent.
REQUIRED_ML_OUTPUTS = {
    "cluster_id",
    "risk_tier",
    "final_risk_score",
    "top_drivers",
    "is_anomaly"
}

# Valeurs autorisées pour risk_tier.
VALID_TIERS = {
    "low",
    "medium",
    "high"
}


# ------------------------------------------------------------
# 7) Vérification du contrat
# ------------------------------------------------------------

# On vérifie s’il manque une feature obligatoire.
missing_feats = REQUIRED_FEATURES - features_payload.keys()

# On vérifie s’il manque une sortie ML obligatoire.
missing_ml = REQUIRED_ML_OUTPUTS - ml_outputs_payload.keys()

# On vérifie si risk_tier contient une valeur autorisée.
invalid_tier = ml_outputs_payload["risk_tier"] not in VALID_TIERS


# ------------------------------------------------------------
# 8) Affichage du résultat de validation
# ------------------------------------------------------------

print()
print("Validation du contrat :")

# Vérifie que toutes les features attendues sont présentes.
print(
    f"  features complet     : "
    f"{'OK' if not missing_feats else 'MANQUE ' + str(missing_feats)}"
)

# Vérifie que toutes les sorties ML attendues sont présentes.
print(
    f"  ml_outputs complet   : "
    f"{'OK' if not missing_ml else 'MANQUE ' + str(missing_ml)}"
)

# Vérifie que risk_tier vaut bien low, medium ou high.
print(
    f"  risk_tier valide     : "
    f"{'OK' if not invalid_tier else 'INVALIDE: ' + repr(ml_outputs_payload['risk_tier'])}"
)

# Vérifie que top_drivers contient au moins un facteur explicatif.
print(
    f"  top_drivers non-vide : "
    f"{'OK' if len(ml_outputs_payload['top_drivers']) > 0 else 'VIDE'}"
)


Simulation pour msisdn = 21697448723
  STATE         = ACTIVE
  cluster_name  = Standard
  risk_label    = Sans risque (level 1)

features (attendu par profiling_node) :
  TOTAL_OUTSTANDING_AMOUNT       = 25.0
  AVG_REIMBURSE_RATIO            = 0.736842105263158
  NB_SOS                         = 19.0

ml_outputs (attendu par profiling/explainability/decision/message) :
  cluster_id         = 0
  risk_tier          = low
  final_risk_score   = 0.09953543299896912
  top_drivers:
    - TOTAL_OUTSTANDING_AMOUNT   value=      25.0 z=+2.58
    - debt_to_credit             value=     6.552 z=+0.90
    - AVG_CREDIT_AMOUNT          value=     3.816 z=+0.76
    - full_repayer               value=       0.0 z=-0.73
  is_anomaly         = False

Validation du contrat :
  features complet     : OK
  ml_outputs complet   : OK
  risk_tier valide     : OK
  top_drivers non-vide : OK


## Limites du modèle

Ce pipeline ne constitue pas un modèle supervisé de prédiction du défaut de paiement.

Il permet d’identifier des profils comportementaux et de produire une lecture de risque métier explicable à partir des données disponibles. Les résultats doivent donc être considérés comme un outil d’aide à la décision, et non comme une décision automatique définitive.

La Risk Ladder repose sur des règles métier et des signaux comportementaux, mais elle ne remplace pas une validation métier ou un contrôle humain dans les cas sensibles.

Une amélioration future consisterait à entraîner un modèle supervisé si l’entreprise fournit un historique fiable des impayés, relances, recouvrements, churn, contentieux ou remboursements après relance.


## Validation finale export

* Cette étape vérifie que le fichier `clients_segmented.csv` a bien été généré.

* Elle contrôle que le fichier est conforme au contrat attendu par le backend et l’agent décisionnel.

* Les vérifications portent sur :

  * l’existence du fichier CSV ;
  * le nombre de lignes et de colonnes ;
  * l’absence de doublons sur `msisdn` ;
  * la présence de toutes les colonnes obligatoires ;
  * la validité de `risk_tier` ;
  * la présence de scores de risque entre 0 et 1 ;
  * la présence de `top_drivers` non vides.

* Si une condition n’est pas respectée, le code arrête l’exécution avec une erreur claire.

* Si toutes les validations passent, le fichier est considéré comme conforme pour l’intégration backend / agent décisionnel.


In [42]:

# ------------------------------------------------------------
# Validation finale export
# ------------------------------------------------------------

from pathlib import Path
import os
import pandas as pd


# ------------------------------------------------------------
# 1) Chemin du fichier exporté
# ------------------------------------------------------------

# Dossier contenant le fichier CSV final.
# Si EXPORT_DIR n'est pas déjà défini dans ton notebook,
# tu peux le définir comme ceci :
EXPORT_DIR = Path(os.environ.get('BAD_DEBTS_EXPORT_DIR', r'C:\\Users\\benab\\OneDrive\\Bureau\\PFE TT\\PLATEFORME TT\\backend\\runtime_data\\ml_imports\\28'))

# Chemin complet du fichier clients_segmented.csv.
CLIENTS_FILE = Path(os.environ.get('BAD_DEBTS_CLIENTS_SEGMENTED_FILE', str(EXPORT_DIR / 'clients_segmented.csv')))


# ------------------------------------------------------------
# 2) Colonnes obligatoires attendues dans le CSV
# ------------------------------------------------------------

# Ces colonnes représentent le contrat de sortie attendu
# par le backend et l’agent décisionnel.
required_columns = [
    "msisdn",
    "STATE",

    "cluster_id",
    "cluster_name",

    "risk_level",
    "risk_label",
    "risk_tier",

    "final_risk_score",
    "risk_score_raw",

    "is_anomaly",
    "anomaly_score",

    "top_drivers",

    "AVG_CREDIT_AMOUNT",
    "AVG_REIMBURSE_RATIO",
    "AVG_DAYS_SINCE_CREDIT",

    "TOTAL_OUTSTANDING_AMOUNT",
    "NB_SOS",

    "debt_to_credit",
    "credit_intensity",
    "tenure_days",

    "has_debt",
    "uses_sos",
    "never_repaid",
    "full_repayer",
    "is_dormant_like",
]


# ------------------------------------------------------------
# 3) Vérification de l’existence du fichier
# ------------------------------------------------------------

# On vérifie que le CSV existe bien.
# Si le fichier est introuvable, le code s’arrête avec un message clair.
assert CLIENTS_FILE.exists(), f"Export introuvable : {CLIENTS_FILE}"


# ------------------------------------------------------------
# 4) Lecture du CSV exporté
# ------------------------------------------------------------

# On lit le fichier CSV final.
export_df = pd.read_csv(CLIENTS_FILE)


# ------------------------------------------------------------
# 5) Vérification des dimensions
# ------------------------------------------------------------

# Nombre attendu de lignes.
# Ici, le dataset final contient 9 748 clients.
EXPECTED_ROWS = 9748

# Nombre attendu de colonnes.
# On le calcule directement à partir de required_columns.
EXPECTED_COLS = len(required_columns)

# Vérification du nombre de lignes et colonnes.
assert export_df.shape == (EXPECTED_ROWS, EXPECTED_COLS), (
    f"Dimensions inattendues : {export_df.shape}"
)


# ------------------------------------------------------------
# 6) Vérification des doublons client
# ------------------------------------------------------------

# Chaque msisdn doit apparaître une seule fois.
assert not export_df["msisdn"].duplicated().any(), (
    "Doublons msisdn détectés"
)


# ------------------------------------------------------------
# 7) Vérification des colonnes obligatoires
# ------------------------------------------------------------

# On identifie les colonnes obligatoires absentes du CSV.
missing_cols = [
    c for c in required_columns
    if c not in export_df.columns
]

# S’il manque une colonne, le fichier n’est pas conforme.
assert not missing_cols, (
    f"Colonnes obligatoires manquantes : {missing_cols}"
)


# ------------------------------------------------------------
# 8) Vérification des valeurs de risk_tier
# ------------------------------------------------------------

# risk_tier doit contenir uniquement low, medium ou high.
valid_tiers = {"low", "medium", "high"}

assert set(export_df["risk_tier"].dropna().unique()).issubset(valid_tiers), (
    "risk_tier invalide"
)


# ------------------------------------------------------------
# 9) Vérification du score final
# ------------------------------------------------------------

# final_risk_score doit toujours être compris entre 0 et 1.
assert export_df["final_risk_score"].between(0, 1).all(), (
    "final_risk_score hors [0, 1]"
)


# ------------------------------------------------------------
# 10) Vérification de top_drivers
# ------------------------------------------------------------

# top_drivers ne doit pas contenir de valeurs manquantes.
assert export_df["top_drivers"].notna().all(), (
    "top_drivers contient des valeurs manquantes"
)

# top_drivers ne doit pas contenir de chaînes vides.
assert (export_df["top_drivers"].astype(str).str.strip() != "").all(), (
    "top_drivers contient des chaînes vides"
)


# ------------------------------------------------------------
# 11) Résultat final
# ------------------------------------------------------------

print("Export conforme pour intégration backend / agent décisionnel.")
print("Shape :", export_df.shape)


# Aperçu des colonnes principales utilisées par l’agent décisionnel.
display(
    export_df[
        [
            "msisdn",
            "cluster_name",
            "risk_tier",
            "final_risk_score",
            "is_anomaly",
            "top_drivers",
        ]
    ].head(10)
)



Export conforme pour intégration backend / agent décisionnel.
Shape : (9748, 25)
        msisdn cluster_name risk_tier  final_risk_score  is_anomaly                                        top_drivers
0  21696330926     Standard       low             0.146       False  [{"feature": "full_repayer", "value": 0.0, "z_...
1  21699807992     Standard    medium             0.331       False  [{"feature": "reimburse_ratio", "value": 0.4, ...
2  21699128564   Bon-payeur       low             0.087       False  [{"feature": "has_debt", "value": 0.0, "z_scor...
3  21696139573     Standard      high             0.246        True  [{"feature": "never_repaid", "value": 1.0, "z_...
4  21696480322   Bon-payeur       low             0.146       False  [{"feature": "has_debt", "value": 0.0, "z_scor...
5  21697035088   Bon-payeur       low             0.069       False  [{"feature": "has_debt", "value": 0.0, "z_scor...
6  21697050056   Bon-payeur       low             0.117       False  [{"feature": "has

## Génération des rapports de contrôle

* Cette étape génère des rapports texte dans le dossier `machine_learning/reports`.

* L’objectif est de garder une trace vérifiable :

  * de la qualité du dataset source ;
  * de la conformité du fichier final exporté.

* Deux rapports sont produits :

1. `data_quality_report.txt`

   * contient un audit synthétique du dataset source ;
   * affiche le nombre de lignes, colonnes, doublons MSISDN ;
   * liste les valeurs manquantes ;
   * vérifie les valeurs négatives sur les colonnes financières ;
   * contrôle que `AVG_REIMBURSE_RATIO` reste dans des bornes cohérentes.

2. `clients_segmented_validation_report.txt`

   * contient un résumé du fichier final `clients_segmented.csv` ;
   * affiche le nombre de lignes et colonnes ;
   * vérifie les colonnes obligatoires ;
   * affiche les doublons `msisdn` ;
   * résume la répartition des `risk_tier`, `cluster_name` et anomalies ;
   * donne les statistiques du score final.

* Ces rapports permettent de documenter la qualité des données et de justifier que l’export final est conforme pour l’intégration backend / agent décisionnel.


In [43]:

# ------------------------------------------------------------
# Génération des rapports de contrôle
# ------------------------------------------------------------

# Dossier où seront stockés les rapports texte.
REPORTS_DIR = ML_DIR / "reports"

# Création du dossier s’il n’existe pas.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1) Rapport qualité source synthétique
# ============================================================

# Liste qui va contenir toutes les lignes du rapport qualité.
quality_lines = []

# Titre du rapport.
quality_lines.append("=" * 90)
quality_lines.append("Audit qualité synthétique du dataset Machine Learning Bad Debts")
quality_lines.append("=" * 90)

# Dimensions du dataset source.
quality_lines.append(f"Lignes source  : {df_raw.shape[0]}")
quality_lines.append(f"Colonnes source: {df_raw.shape[1]}")

# Vérification des doublons MSISDN si la colonne existe.
quality_lines.append(
    f"Doublons MSISDN: "
    f"{df_raw['MSISDN'].duplicated().sum() if 'MSISDN' in df_raw.columns else 'MSISDN absent'}"
)

quality_lines.append("")


# ------------------------------------------------------------
# 1.1 Valeurs manquantes
# ------------------------------------------------------------

quality_lines.append("Valeurs manquantes non nulles :")

# Calcul du nombre de valeurs manquantes par colonne.
missing_source = df_raw.isna().sum()

# On garde uniquement les colonnes qui ont au moins une valeur manquante.
missing_source = missing_source[missing_source > 0]

# Si aucune valeur manquante n’est détectée.
if missing_source.empty:
    quality_lines.append("Aucune valeur manquante détectée.")

# Sinon, on ajoute le détail dans le rapport.
else:
    quality_lines.append(str(missing_source))

quality_lines.append("")


# ------------------------------------------------------------
# 1.2 Contrôles financiers
# ------------------------------------------------------------

quality_lines.append("Contrôles financiers :")

# Colonnes financières à contrôler.
financial_cols = [
    "AVG_CREDIT_AMOUNT",
    "AVG_CREDIT_FEE",
    "AVG_REIMBURSED_AMOUNT",
    "AVG_FEE_REIMBURSED",
    "AVG_REIMBURSE_RATIO",
    "AVG_DAYS_SINCE_CREDIT",
    "TOTAL_OUTSTANDING_AMOUNT",
    "TOTAL_OUTSTANDING_FEE",
    "NB_SOS",
]

# Pour chaque colonne financière présente dans df_raw,
# on vérifie le nombre de valeurs négatives.
for col in financial_cols:

    if col in df_raw.columns:

        # Conversion en numérique pour éviter les erreurs de type.
        values = pd.to_numeric(df_raw[col], errors="coerce")

        # Comptage des valeurs négatives.
        quality_lines.append(
            f"{col}: valeurs négatives = {int((values < 0).sum())}"
        )


# Contrôle spécifique du ratio de remboursement.
# Normalement, un ratio doit être entre 0 et 1.
if "AVG_REIMBURSE_RATIO" in df_raw.columns:

    ratio = pd.to_numeric(df_raw["AVG_REIMBURSE_RATIO"], errors="coerce")

    quality_lines.append(f"AVG_REIMBURSE_RATIO < 0 : {int((ratio < 0).sum())}")
    quality_lines.append(f"AVG_REIMBURSE_RATIO > 1 : {int((ratio > 1).sum())}")

quality_lines.append("")


# Conclusion du rapport qualité.
quality_lines.append(
    "Conclusion : dataset source exploitable pour segmentation comportementale non supervisée."
)


# Écriture du rapport qualité dans un fichier texte.
(REPORTS_DIR / "data_quality_report.txt").write_text(
    "\n".join(map(str, quality_lines)),
    encoding="utf-8"
)


# ============================================================
# 2) Rapport conformité export
# ============================================================

# Liste qui va contenir les lignes du rapport de validation export.
validation_lines = []

# Titre du rapport.
validation_lines.append("=" * 90)
validation_lines.append("Validation du fichier clients_segmented.csv")
validation_lines.append("=" * 90)

# Dimensions du fichier exporté.
validation_lines.append(f"Lignes  : {export_df.shape[0]}")
validation_lines.append(f"Colonnes: {export_df.shape[1]}")
validation_lines.append("")


# ------------------------------------------------------------
# 2.1 Colonnes et doublons
# ------------------------------------------------------------

# Liste des colonnes obligatoires manquantes.
validation_lines.append(
    "Colonnes obligatoires manquantes : "
    + (", ".join(missing_cols) if missing_cols else "aucune")
)

# Nombre de doublons sur msisdn.
validation_lines.append(
    f"Doublons msisdn : {int(export_df['msisdn'].duplicated().sum())}"
)

validation_lines.append("")


# ------------------------------------------------------------
# 2.2 Répartition des tiers de risque
# ------------------------------------------------------------

validation_lines.append("Répartition risk_tier :")
validation_lines.append(str(export_df["risk_tier"].value_counts()))
validation_lines.append("")


# ------------------------------------------------------------
# 2.3 Répartition des clusters
# ------------------------------------------------------------

validation_lines.append("Répartition cluster_name :")
validation_lines.append(str(export_df["cluster_name"].value_counts()))
validation_lines.append("")


# ------------------------------------------------------------
# 2.4 Répartition des anomalies
# ------------------------------------------------------------

validation_lines.append("Anomalies :")
validation_lines.append(str(export_df["is_anomaly"].value_counts()))
validation_lines.append("")


# ------------------------------------------------------------
# 2.5 Statistiques du score final
# ------------------------------------------------------------

validation_lines.append("Score final :")
validation_lines.append(
    str(export_df["final_risk_score"].agg(["min", "mean", "max"]))
)
validation_lines.append("")


# Conclusion du rapport de validation.
validation_lines.append(
    "VALIDATION : fichier clients_segmented.csv conforme pour intégration agent décisionnel / backend."
)


# Écriture du rapport de validation export.
(REPORTS_DIR / "clients_segmented_validation_report.txt").write_text(
    "\n".join(map(str, validation_lines)),
    encoding="utf-8"
)


# ============================================================
# 3) Affichage final
# ============================================================

print("Rapports générés dans :", REPORTS_DIR)
print(" - data_quality_report.txt")
print(" - clients_segmented_validation_report.txt")



Rapports générés dans : C:\Users\benab\OneDrive\Bureau\PFE TT\PLATEFORME TT\machine_learning\reports
 - data_quality_report.txt
 - clients_segmented_validation_report.txt


## Résumé exécutif des résultats

* Cette cellule produit un résumé final des résultats exportés.

* Elle permet de vérifier rapidement :

  * la taille du fichier final ;
  * la répartition des clusters ;
  * la répartition des niveaux de risque simplifiés ;
  * le nombre d’anomalies détectées ;
  * les statistiques du score final ;
  * quelques exemples de clients.

* L’objectif est de donner une vision synthétique du résultat final du pipeline ML.

* Cette cellule ne modifie pas les données.

* Elle sert uniquement à afficher les résultats principaux de manière claire.


In [44]:

# ------------------------------------------------------------
# Résumé exécutif des résultats
# ------------------------------------------------------------

# Message d'introduction du résumé.
print("Résumé exécutif calculé")
print("=" * 80)


# ------------------------------------------------------------
# 1) Taille du fichier final
# ------------------------------------------------------------

# Affiche le nombre de lignes et de colonnes du fichier exporté.
print(
    f"Shape export final : "
    f"{export_df.shape[0]} lignes x {export_df.shape[1]} colonnes"
)


# ------------------------------------------------------------
# 2) Répartition des clusters
# ------------------------------------------------------------

# Affiche le nombre de clients dans chaque cluster ou segment.
#
# Exemple :
# - Bon-payeur
# - Standard
# - ON-HOLD
# - SUSPENDED
# - DISCONNECTED
print("\nRépartition des clusters :")
display(export_df["cluster_name"].value_counts())


# ------------------------------------------------------------
# 3) Répartition des niveaux de risque
# ------------------------------------------------------------

# Affiche la distribution des risk_tier :
# - low
# - medium
# - high
#
# Ce sont les niveaux simplifiés utilisés par l’agent décisionnel.
print("\nRépartition des niveaux de risque :")
display(export_df["risk_tier"].value_counts())


# ------------------------------------------------------------
# 4) Nombre d’anomalies détectées
# ------------------------------------------------------------

# is_anomaly vaut True si le client est détecté comme anomalie.
# Comme True vaut 1 et False vaut 0, sum() donne le nombre d’anomalies.
print("\nNombre d'anomalies détectées par Isolation Forest :")
print(int(export_df["is_anomaly"].sum()))


# ------------------------------------------------------------
# 5) Statistiques du score final de risque
# ------------------------------------------------------------

# Affiche le minimum, la moyenne et le maximum du score final.
#
# Le score final doit normalement être compris entre 0 et 1.
print("\nScore final de risque :")
display(export_df["final_risk_score"].agg(["min", "mean", "max"]))


# ------------------------------------------------------------
# 6) Exemples de clients
# ------------------------------------------------------------

# Affiche les premières lignes avec les colonnes principales.
#
# Cela permet de vérifier rapidement que les résultats sont lisibles.
print("\nExemples clients :")
display(
    export_df[
        [
            "msisdn",
            "cluster_name",
            "risk_tier",
            "final_risk_score",
            "is_anomaly",
            "top_drivers",
        ]
    ].head(10)
)



Résumé exécutif calculé
Shape export final : 9748 lignes x 25 colonnes

Répartition des clusters :
cluster_name
Standard        6602
Bon-payeur      2606
SUSPENDED        497
DISCONNECTED      28
ON-HOLD           15
Name: count, dtype: int64

Répartition des niveaux de risque :
risk_tier
low       5964
medium    2105
high      1679
Name: count, dtype: int64

Nombre d'anomalies détectées par Isolation Forest :
277

Score final de risque :
min    0.015
mean   0.275
max    0.988
Name: final_risk_score, dtype: float64

Exemples clients :
        msisdn cluster_name risk_tier  final_risk_score  is_anomaly                                        top_drivers
0  21696330926     Standard       low             0.146       False  [{"feature": "full_repayer", "value": 0.0, "z_...
1  21699807992     Standard    medium             0.331       False  [{"feature": "reimburse_ratio", "value": 0.4, ...
2  21699128564   Bon-payeur       low             0.087       False  [{"feature": "has_debt", "value":

## Validation métier des clusters

* Cette cellule vérifie si les clusters obtenus sont cohérents d’un point de vue métier.

* Pour chaque `cluster_name`, on calcule la moyenne de plusieurs variables importantes :

  * montant moyen du crédit ;
  * taux moyen de remboursement ;
  * dette restante ;
  * nombre d’utilisations SOS ;
  * ratio dette/crédit ;
  * score final de risque.

* Le tableau permet de comparer les profils moyens des clusters.

* Par exemple, un cluster `Bon-payeur` doit normalement avoir :

  * un meilleur taux de remboursement ;
  * une dette plus faible ;
  * un score final de risque plus bas.

* À l’inverse, les segments plus risqués doivent avoir un score final moyen plus élevé.

* Le tri par `final_risk_score` permet de lire les clusters du moins risqué au plus risqué.


In [45]:
# ------------------------------------------------------------
# Validation métier des clusters
# ------------------------------------------------------------

# Variables utilisées pour comparer les clusters.
#
# Ces colonnes permettent de comprendre le profil métier moyen
# de chaque cluster.
cluster_validation_cols = [
    "AVG_CREDIT_AMOUNT",
    "AVG_REIMBURSE_RATIO",
    "TOTAL_OUTSTANDING_AMOUNT",
    "NB_SOS",
    "debt_to_credit",
    "final_risk_score",
]


# ------------------------------------------------------------
# Calcul du profil moyen par cluster
# ------------------------------------------------------------

cluster_business_validation = (
    export_df

    # On regroupe les clients par cluster ou segment.
    .groupby("cluster_name")[cluster_validation_cols]

    # On calcule la moyenne de chaque variable pour chaque cluster.
    .mean()

    # On arrondit les résultats pour améliorer la lisibilité.
    .round(3)

    # On trie les clusters du moins risqué au plus risqué.
    .sort_values("final_risk_score")
)


# Affichage du tableau de validation métier.
display(cluster_business_validation)

              AVG_CREDIT_AMOUNT  AVG_REIMBURSE_RATIO  TOTAL_OUTSTANDING_AMOUNT  NB_SOS  debt_to_credit  final_risk_score
cluster_name                                                                                                            
Bon-payeur                2.349                1.000                     0.000   4.395           0.000             0.236
Standard                  2.069                0.716                     7.280  22.259           4.057             0.242
ON-HOLD                   0.916                0.067                     1.147   1.467           1.333             0.743
SUSPENDED                 0.833                0.565                     1.302   5.889           1.640             0.872
DISCONNECTED              4.040                0.643                     5.644   9.000           2.346             0.958


## Features brutes vs features construites

Les variables brutes décrivent directement les montants, ratios et volumes fournis dans le dataset source. Les features construites ajoutent une lecture métier plus expressive : intensité d’usage, dette relative, ancienneté client, comportement de remboursement et signaux binaires simples.

Ces features construites facilitent l’interprétation des clusters et du Risk Ladder.

In [46]:
raw_features = [
    "AVG_CREDIT_AMOUNT",
    "AVG_CREDIT_FEE",
    "AVG_REIMBURSED_AMOUNT",
    "AVG_FEE_REIMBURSED", 
    "AVG_REIMBURSE_RATIO",
    "AVG_DAYS_SINCE_CREDIT",
    "TOTAL_OUTSTANDING_AMOUNT",
    "TOTAL_OUTSTANDING_FEE",
    "NB_SOS",
]

final_features = [
    "AVG_CREDIT_AMOUNT",
    "AVG_REIMBURSE_RATIO",
    "AVG_DAYS_SINCE_CREDIT",
    "TOTAL_OUTSTANDING_AMOUNT",
    "NB_SOS",
    "debt_to_credit",
    "credit_intensity",
    "tenure_days",
    "has_debt",
    "uses_sos",
    "never_repaid",
    "full_repayer",
    "is_dormant_like",
]

print(f"Nombre de variables brutes : {len(raw_features)}")
print(f"Nombre de features finales : {len(final_features)}")
print("\nFeatures finales :")
for feature in final_features:
    print(f" - {feature}")

Nombre de variables brutes : 9
Nombre de features finales : 13

Features finales :
 - AVG_CREDIT_AMOUNT
 - AVG_REIMBURSE_RATIO
 - AVG_DAYS_SINCE_CREDIT
 - TOTAL_OUTSTANDING_AMOUNT
 - NB_SOS
 - debt_to_credit
 - credit_intensity
 - tenure_days
 - has_debt
 - uses_sos
 - never_repaid
 - full_repayer
 - is_dormant_like
